# AI Financial Digital Twin — Master Runner
## SECTION 0 — Prototype disclaimer
Synthetic data only. Prototype only. Not regulatory stress testing, financial advice, or an approved bank risk model.

### Reader guide
- **Section 7 — scenario set:** runs a selected list of scenarios independently and compares one row per run. It does not combine them.
- **Cloud Region A eight-hour analysis:** runs `cloud_region_a_8hr` scenario independently. It is separate from Section 7 and separate from the combined stress.
- **Section 8 onward — flagship combined stress:** runs `combined_stress` scenario. The LCR bridge, Monte Carlo, management actions, AI executive explanation, and CEO dashboard primarily describe this combined-stress run.
- **LLM role:** explains validated Python output only; it does not calculate impacts.

## SECTION 1 — Colab setup
**Purpose:** connect the notebook to the repository in Google Drive or detect a local checkout.

**Input:** `PROJECT_ROOT`. Change only this path when your Drive folder differs. **Output:** active project and source paths. No scenario is run here.

In [ ]:
# Purpose: This section sets up the Python environment, connects to Google Drive (if in Colab), and defines the project root and source paths.
# It ensures that necessary modules can be imported and that the notebook operates from the correct directory.

import os, sys, pathlib

# Try to mount Google Drive if running in a Colab environment.
# This allows access to files stored in Google Drive.
# Interdependency: Requires 'google.colab' library.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    # If not in Colab, print a message indicating local runtime.
    print('Local runtime detected; Drive mount skipped.')

# === CHANGE ONLY THIS VARIABLE IN COLAB IF NEEDED ===
# Input: PROJECT_ROOT (str) - The path to the root directory of the project.
# If running in Colab, it's typically '/content/drive/MyDrive/ai-financial-digital-twin'.
# If running locally, it attempts to determine the root based on the current working directory.
PROJECT_ROOT = '/content/drive/MyDrive/ai-financial-digital-twin'
if not os.path.exists(PROJECT_ROOT):
    # Adjust PROJECT_ROOT for local execution if the Colab path doesn't exist.
    PROJECT_ROOT = str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())

# Define SRC_PATH, the directory containing custom Python modules.
# Interdependency: Depends on PROJECT_ROOT.
SRC_PATH = os.path.join(PROJECT_ROOT, 'src')
# Add SRC_PATH to sys.path so Python can find and import modules from this directory.
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# Change the current working directory to PROJECT_ROOT.
# This ensures that relative file paths within the project work correctly.
os.chdir(PROJECT_ROOT)
# Output: Prints the determined project root path.
print('Project root:', PROJECT_ROOT)

# Check for GPU availability using PyTorch.
# Interdependency: Requires the 'torch' library.
# This is an optional check, as GPU is not strictly required for the notebook's core functionality.
try:
    import torch
    # Output: Prints whether a GPU is available.
    print('GPU available:', torch.cuda.is_available())
except ImportError:
    # If PyTorch is not installed, print a message.
    print('GPU library not installed (GPU is not required).')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/ai-financial-digital-twin
GPU available: False


## SECTION 2 — Install dependencies
**Purpose:** check and install the declared Python libraries. **Input:** `requirements.txt`. **Output:** a ready runtime. No bank data or scenario result is changed.

In [ ]:
import subprocess
import sys
import os

# Purpose: This section ensures that all necessary Python libraries for the project are installed.
# It checks for required packages and installs any missing ones using pip and the requirements.txt file.

# Input:
# - `required` (list of str): A hardcoded list of Python package names that are essential for the notebook.
# - `PROJECT_ROOT` (str): Obtained from SECTION 1, used to locate `requirements.txt`.
# - `requirements.txt` file: Specifies additional dependencies and their versions.

# Interdependencies:
# - Relies on `PROJECT_ROOT` being correctly defined in SECTION 1.
# - Requires `pip` to be available in the execution environment.

# List of essential Python packages.
required = ['numpy','pandas','scipy','networkx','simpy','plotly','yaml','openai']
missing = []

# Check if each required package is installed.
for package in required:
    try:
        __import__(package)
    except ImportError:
        # If a package is not found, add it to the 'missing' list.
        missing.append(package)

# If there are any missing packages, install them using pip and requirements.txt.
if missing:
    print(f'Missing packages: {missing}. Installing from requirements.txt...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', os.path.join(PROJECT_ROOT, 'requirements.txt')])

# Output:
# - Prints 'Dependencies ready.' once all required packages are confirmed to be installed.
# - (Implicit) The Python environment is configured with all necessary libraries, allowing subsequent cells to import and use them.
print('Dependencies ready.')

Dependencies ready.


## SECTION 3 — Import modules
**Purpose:** load the Digital Twin engines, validation helpers, decision tools, and visualizations. **Output:** callable Python functions; no simulation is run.

In [ ]:
import json
import pandas as pd
from IPython.display import display

# Purpose: This section imports all necessary custom modules and functions developed for the Digital Twin project.
# These modules contain the core logic for data generation, scenario simulation, dependency analysis, Monte Carlo simulation,
# management action analysis, AI explanations, and visualizations.

# Input:
# - No direct user input for this cell. It relies on the availability of the custom Python packages
#   located in the 'src' directory, which was added to the Python path in SECTION 1.

# Interdependencies:
# - Relies on the 'digital_twin' package structure and its submodules (e.g., data_generator, scenario_engine, etc.).
# - Assumes these packages have been successfully installed or made available in the Python path (handled in SECTION 1 and SECTION 2).
# - The functions and classes imported here will be used in subsequent sections of the notebook.

# Imports from standard libraries:
# - json: For working with JSON data, especially for exporting and handling structured data.
# - pandas: For data manipulation and analysis, primarily with DataFrames.
# - IPython.display.display: For rich display of Python objects in Jupyter/Colab notebooks.

# Imports from custom 'digital_twin' modules:
from digital_twin.data_generator import generate_virtual_bank # Used in SECTION 4 to create the synthetic bank.
from digital_twin.scenario_engine import ScenarioEngine # Core engine for running scenarios, used in SECTION 5, 7, 8, 9, 10, 13.
from digital_twin.dependency_graph import build_dependency_graph, centrality_table, cloud_concentration_metrics, critical_nodes, single_points_of_failure, top_propagation_paths # Used in SECTION 6 and CLOUD REGION FAILURE sections for graph analysis.
from digital_twin.monte_carlo import run_monte_carlo, summarize_monte_carlo, metric_percentiles, breach_probability_table, explain_probability_extremes, operational_breach_diagnostics # Used in SECTION 9 and MONTE CARLO OPERATIONAL BREACH VALIDATION for stochastic simulation and analysis.
from digital_twin.action_engine import compare_actions, attribute_management_actions, analyze_management_strategies # Used in SECTION 10 and MANAGEMENT RESPONSE DECISION LAB for management action analysis.
from digital_twin.ai_explainer import explain_results, answer_question, build_executive_context, payload_size_comparison, explain_executive_context # Used in SECTION 11 and SECTION 13 for AI-driven explanations.
from digital_twin.visualizations import * # Imports all visualization functions, used throughout the notebook (e.g., SECTION 5, 6, 7, 8, 9, CLOUD RESILIENCE DIGITAL TWIN).

# Output:
# - Prints 'Digital Twin imports succeeded.' upon successful execution.
# - Makes all imported functions and classes available for use in subsequent cells, forming the foundation of the Digital Twin analysis.

## SECTION 4 — Generate synthetic bank
**Purpose:** create the reproducible synthetic bank used by every later scenario.

**Input:** baseline YAML, risk-limit YAML, generated application/customer/counterparty data, and seed 42.

**Output:** `bank`, `engine`, and the shared NetworkX graph.

In [ ]:
# Purpose: This cell generates a synthetic virtual bank, which serves as the base data for all subsequent scenarios and simulations.
# It ensures reproducibility by using a fixed random seed (42).

# Input:
# - seed (int): A numerical seed for random number generation to ensure that the generated bank data is consistent across runs.
# - The function `generate_virtual_bank` internally loads configuration files (baseline YAML, risk-limit YAML)
#   and generates synthetic application, customer, and counterparty data.
# Interdependencies:
# - Relies on the `generate_virtual_bank` function imported from `digital_twin.data_generator` (in SECTION 3).

bank = generate_virtual_bank(seed=42)

# Output:
# - `bank` (Bank object): An instance of the Bank class containing all generated financial and operational data.
#   This object is critical for initializing the `ScenarioEngine` and all subsequent analyses.
# - Prints and displays various Pandas DataFrames representing different aspects of the bank's structure:
#   - Balance sheet
#   - Customer segments
#   - FX exposures
#   - Counterparties
#   - Applications
#   - Infrastructure
# These displays provide a high-level overview of the synthetic bank's composition.
for label, frame in [('Balance sheet', bank.balance_sheet), ('Customer segments', bank.customer_segments), ('FX exposures', bank.fx_exposures), ('Counterparties', bank.counterparties), ('Applications', bank.applications), ('Infrastructure', bank.infrastructure)]:
    print('\n', label); display(frame)


 Balance sheet


,item,category,amount_bn
0,cash_reserves,asset,10.0
1,securities,asset,18.0
2,customer_loans,asset,72.0
3,retail_deposits,liability,42.0
4,sme_deposits,liability,12.0
5,corporate_deposits,liability,24.0
6,wholesale_funding,liability,12.0
7,other_liabilities,liability,2.0
8,cet1_capital,capital,8.0



 Customer segments


,segment,deposits_bn,baseline_outflow_rate,withdrawal_sensitivity,credit_utilisation,outage_sensitivity,customers
0,Retail,42.0,0.05,0.7,0.35,0.8,4200000
1,SME,12.0,0.10,1.0,0.55,1.0,180000
2,Corporate,20.0,0.15,1.4,0.70,1.3,8000
3,Private Banking,4.0,0.08,1.2,0.45,1.1,30000



 FX exposures


...

,counterparty_id,ead_bn,pd,lgd,rating,sector,collateral_ratio
0,CP-001,1.449632,0.0005,0.557579,AAA,Financials,0.385271
1,CP-002,1.330920,0.0010,0.592687,AA,Technology,0.161215
2,CP-003,1.429767,0.0030,0.338434,A,Manufacturing,0.610942
3,CP-004,0.948098,0.0100,0.578029,BBB,Energy,0.341019
4,CP-005,1.247991,0.0350,0.433024,BB,Property,0.634795
5,CP-006,0.602220,0.0800,0.319145,B,Retail,0.460480
6,CP-007,1.532828,0.0005,0.527426,AAA,Financials,0.510582
7,CP-008,0.799515,0.0010,0.567936,AA,Technology,0.730954
8,CP-009,1.456494,0.0030,0.440016,A,Manufacturing,0.226515
9,CP-010,0.317896,0.0100,0.504915,BBB,Energy,0.200288



 Applications


,application,service,availability_target,primary_region,backup_region,backup_mode,failover_time_minutes,normal_capacity_pct,backup_capacity_pct,criticality
0,Core Banking,Banking,0.990,Primary Data Centre,Backup Data Centre,active-passive,60.0,100,100,Critical
1,Domestic Payments,Payments,0.995,Cloud Region A,Cloud Region B,warm standby,180.0,100,70,Critical
2,Cross-Border Payments,Payments,0.990,Cloud Region A,None,none,NaN,100,0,Critical
3,Mobile Banking,Channels,0.980,Primary Data Centre,Backup Data Centre,active-passive,30.0,100,100,High
4,Internet Banking,Channels,0.980,Primary Data Centre,Backup Data Centre,active-passive,30.0,100,100,High
5,Treasury Platform,Markets,0.990,Primary Data Centre,Backup Data Centre,warm standby,60.0,100,90,Critical
6,Identity Service,Security,0.995,Cloud Region A,Cloud Region B,hot standby,30.0,100,90,Critical
7,Fraud Monitoring,Security,0.990,Primary Data Centre,Backup Data Centre,active-passive,30.0,100,100,High
8,Credit Risk Engine,Risk,0.980,Primary Data Centre,Backup Data Centre,warm standby,120.0,100,80,High
9,Liquidity Risk Platform,Risk,0.980,Primary Data Centre,Backup Data Centre,warm standby,120.0,100,80,High



 Infrastructure


,infrastructure,type,workload_share,primary
0,Cloud Region A,cloud,0.55,True
1,Cloud Region B,cloud,0.30,False
2,Primary Data Centre,data_centre,0.10,True
3,Backup Data Centre,data_centre,0.05,False


## SECTION 5 — Baseline bank health
**Scope:** baseline only—no stress scenario and no management action.

**Purpose:** establish the starting KPIs against which stressed results are compared.


In [ ]:
# Purpose: This cell initializes the ScenarioEngine with the generated bank data and calculates the baseline financial and operational metrics.
# These baseline metrics represent the bank's health under normal, unstressed conditions and serve as a reference point for comparison with stressed scenarios.

# Input:
# - `bank` (Bank object): The synthetic bank object generated in SECTION 4, containing all the necessary data.

# Interdependencies:
# - Relies on the `ScenarioEngine` class imported from `digital_twin.scenario_engine` (in SECTION 3).
# - Uses the `bank` object created in SECTION 4.

# Initialize the ScenarioEngine with the bank object.
engine = ScenarioEngine(bank)

# Calculate the baseline metrics for the bank.
# The 'baseline_metrics' method returns a dictionary of key performance indicators (KPIs).
baseline = engine.baseline_metrics()

# Output:
# - `engine` (ScenarioEngine object): An initialized instance of the ScenarioEngine, which will be used to run various stress scenarios.
# - `baseline` (dict): A dictionary containing the bank's baseline financial and operational metrics (e.g., LCR, CET1 ratio, cash position, etc.).
# - Displays a Pandas DataFrame containing the `baseline` metrics, providing a tabular overview of the bank's initial state.
# - Displays a visualization of the bank's health dashboard, providing a graphical representation of the baseline KPIs.
display(pd.DataFrame([baseline]))
bank_health_dashboard(baseline).show()

,total_estimated_loss_bn,market_loss_bn,credit_loss_bn,operational_loss_bn,liquidity_pnl_loss_bn,cash_position_bn,hqla_bn,lcr,cet1_capital_bn,risk_weighted_assets_bn,...,customers_affected,applications_affected,recovery_time_hours,deposit_outflow_bn,cash_consumed_bn,hqla_consumed_bn,emergency_funding_requirement_bn,funding_cost_bn,asset_liquidation_bn,realised_asset_sale_loss_bn
0,0.0,0.0,0.0,0.0,0.0,10.0,25.3,5.47619,8.0,51.84,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## SECTION 6 — DIGITAL TWIN DEPENDENCY MAP — BASELINE
**Scope:** shared baseline NetworkX model.

**Purpose:** show infrastructure, applications, services, customers, financial positions, and risk metrics before any node fails. The same graph — not an LLM created graph — is used for scenario propagation.

In [ ]:
# Purpose: This cell analyzes the underlying dependency graph of the bank's operations and infrastructure.
# It identifies critical nodes, single points of failure, and provides a visualization of the entire interconnected system.
# This baseline graph is crucial for understanding how shocks propagate through the bank.

# Input:
# - `engine.graph` (NetworkX Graph object): The graph representing the bank's dependencies, built internally by the ScenarioEngine.
#   This graph was initialized when the `ScenarioEngine` was created in SECTION 5.

# Interdependencies:
# - Relies on the `engine` object initialized in SECTION 5.
# - Uses functions from `digital_twin.dependency_graph` (e.g., `centrality_table`, `critical_nodes`, `single_points_of_failure`)
#   and `digital_twin.visualizations` (`dependency_graph_figure`), all imported in SECTION 3.

# Retrieve the NetworkX graph from the ScenarioEngine.
graph = engine.graph  # The exact NetworkX graph used by ScenarioEngine

# Calculate centrality metrics for the graph nodes.
# Centrality measures indicate the importance of a node within the network.
centrality = centrality_table(graph)

# Output:
# - Prints the total number of nodes and dependencies (edges) in the graph.
# - Displays a Pandas DataFrame showing the top 10 nodes by various centrality measures.
# - Prints a list of 'Critical nodes' and 'Single points of failure' as identified by the analysis.
# - Displays an interactive visualization of the bank's entire dependency graph, allowing for visual inspection of connections.
print(f'Nodes: {graph.number_of_nodes():,}')
print(f'Dependencies: {graph.number_of_edges():,}')
display(centrality.head(10))
print('Critical nodes:', critical_nodes(graph))
print('Highest betweenness-centrality nodes:', centrality.nlargest(5, 'betweenness_centrality')['node'].tolist())
print('Single points of failure:', single_points_of_failure(graph))
dependency_graph_figure(graph).show()

Nodes: 30
Dependencies: 36


,node,degree_centrality,betweenness_centrality,downstream_count
0,Deposit Outflows,0.137931,0.078818,4
1,Liquidity Position,0.103448,0.062808,3
2,Domestic Payments,0.206897,0.055419,8
3,SME,0.103448,0.050287,5
4,Capital Position,0.137931,0.032020,1
5,Identity Service,0.172414,0.027094,10
6,Cloud Region A,0.137931,0.018473,15
7,Retail,0.103448,0.017447,5
8,Corporate,0.103448,0.016010,6
9,Internet Banking,0.103448,0.013752,7


Critical nodes: ['Deposit Outflows', 'Liquidity Position', 'Domestic Payments', 'SME', 'Capital Position']
Highest betweenness-centrality nodes: ['Deposit Outflows', 'Liquidity Position', 'Domestic Payments', 'SME', 'Capital Position']
Single points of failure: ['Capital Position', 'Cloud Region A', 'Core Banking', 'Credit Loss', 'Deposit Outflows', 'Domestic Payments', 'FX Exposure', 'Identity Service', 'Liquidity Position', 'Market P&L', 'Mobile Banking', 'Treasury Platform']


## SECTION 7 — Individual stress scenarios
**Analysis scope: SELECTED SCENARIO SET, RUN INDEPENDENTLY.**

The `scenario_names` list controls which YAML scenarios are run. Each scenario starts from the same baseline bank and produces one independent result row. Adding two names to this list compares two separate runs; it does **not** merge their shocks.

**Current set:** `usd_fall`, `deposit_run`, `payment_outage`, `volatility_shock`, `cloud_failure`, and `counterparty_default`. **Output:** `individual_results` and `scenario_table`.

The detailed eight-hour cloud scenario and flagship combined stress are run separately in the next major analyses.

In [ ]:
# Purpose: This cell runs a predefined list of individual stress scenarios, each independently, against the baseline bank.
# It allows for a comparison of how different single shocks impact the bank's metrics.

# Input:
# - `scenario_names` (list of str): A list of scenario YAML file names (e.g., 'usd_fall', 'deposit_run').
#   These names correspond to configuration files defining the specific shocks for each scenario.
# - `engine` (ScenarioEngine object): The initialized scenario engine from SECTION 5, which handles scenario execution.

# Interdependencies:
# - Relies on the `engine` object initialized in SECTION 5.
# - Assumes the existence of scenario configuration files (YAMLs) that the `engine.run()` method can load and execute.
# - Uses the `scenario_comparison` visualization function imported in SECTION 3.

# Define the list of scenario names to be run.
scenario_names = ['usd_fall','deposit_run','payment_outage','volatility_shock','cloud_failure','counterparty_default']

# Execute each scenario independently using a list comprehension.
# Each call to `engine.run(name)` simulates the bank under that specific stress and returns a ScenarioResult object.
individual_results = [engine.run(name) for name in scenario_names]

# Process the results into a Pandas DataFrame for easy viewing and comparison.
# For each result, it extracts the scenario name, its calculated metrics, and the count of risk limit breaches.
scenario_table = pd.DataFrame([{'scenario': r.scenario, **r.metrics, 'breaches': len(r.risk_limit_breaches)} for r in individual_results])

# Output:
# - `individual_results` (list of ScenarioResult objects): A list containing the full results for each run scenario.
# - `scenario_table` (Pandas DataFrame): A summary table of key metrics and risk breaches for each individual scenario run.
# - Displays the `scenario_table` for a quick overview of the impacts.
# - Displays a graphical `scenario_comparison` visualization, providing a visual comparison of key metrics across scenarios.
display(scenario_table)
scenario_comparison(scenario_table).show()

,scenario,total_estimated_loss_bn,market_loss_bn,credit_loss_bn,operational_loss_bn,liquidity_pnl_loss_bn,cash_position_bn,hqla_bn,lcr,cet1_capital_bn,...,applications_affected,recovery_time_hours,deposit_outflow_bn,cash_consumed_bn,hqla_consumed_bn,emergency_funding_requirement_bn,funding_cost_bn,asset_liquidation_bn,realised_asset_sale_loss_bn,breaches
0,USD falls 10 percent,0.360000,0.36,0.000000,0.000000,0.0,9.98560,25.28560,5.456068,7.640000,...,0,0.0,0.00000,0.01440,0.01440,0.0,0.0,0.0,0.0,0
1,Deposit withdrawal stress,0.000000,0.00,0.000000,0.000000,0.0,0.12800,15.42800,1.064587,8.000000,...,0,0.0,9.87200,9.87200,9.87200,0.0,0.0,0.0,0.0,2
2,Payment system unavailable,0.033613,0.00,0.000000,0.033613,0.0,9.68375,24.98375,5.061281,7.966387,...,2,16.0,0.31625,0.31625,0.31625,0.0,0.0,0.0,0.0,2
3,Market volatility doubles,0.120000,0.12,0.000000,0.000000,0.0,9.99520,25.29520,5.469469,7.880000,...,0,0.0,0.00000,0.00480,0.00480,0.0,0.0,0.0,0.0,0
4,Cloud Region A failure,0.064100,0.00,0.000000,0.064100,0.0,9.62056,24.92056,4.984670,7.935900,...,5,13.0,0.37944,0.37944,0.37944,0.0,0.0,0.0,0.0,2
5,Major counterparty default,0.236741,0.00,0.236741,0.000000,0.0,10.00000,25.30000,5.476190,7.763259,...,0,0.0,0.00000,0.00000,0.00000,0.0,0.0,0.0,0.0,0


### DIGITAL TWIN — CLOUD FAILURE PROPAGATION
**Scope:** This is for generic `cloud_failure` result from the Section 7 scenario set and not the dedicated eight-hour cloud scenario ( Which is run later) . Both views use the exact graph held by `ScenarioEngine`. The focused view removes unrelated nodes and retains only simulation-emitted edges that exist in NetworkX.

In [ ]:
# Purpose: This cell visualizes the propagation of a 'cloud_failure' scenario on the bank's dependency graph.
# It presents two views: a full digital twin graph showing all dependencies, and a focused propagation graph
# which highlights only the actual paths of impact from the cloud failure.

# Input:
# - `individual_results` (list of ScenarioResult objects): The results from running individual scenarios in SECTION 7.
# - `scenario_names` (list of str): The names of the scenarios run in SECTION 7.
# - `graph` (NetworkX Graph object): The bank's dependency graph, initialized in SECTION 5.

# Interdependencies:
# - Relies on `individual_results` and `scenario_names` from SECTION 7 to retrieve the specific cloud failure result.
# - Uses the `graph` object from SECTION 5.
# - Leverages visualization functions (`interactive_dependency_graph`, `scenario_propagation_graph_figure`)
#   imported from `digital_twin.visualizations` in SECTION 3.

# Retrieve the result object for the 'cloud_failure' scenario from the list of individual results.
cloud_failure_result = individual_results[scenario_names.index('cloud_failure')]

# Output:
# - Prints descriptive headers for each visualization.
# - Displays an interactive visualization of the FULL DIGITAL TWIN GRAPH, with nodes affected by the cloud failure highlighted.
#   This shows the entire network and where the failure initiated.
# - Displays a FOCUSED PROPAGATION GRAPH, which only shows the specific dependency paths that were activated
#   and impacted by the 'Cloud Region A' failure.
print('FULL DIGITAL TWIN GRAPH — cloud failure impact state')
interactive_dependency_graph(graph, cloud_failure_result.propagation_trace, failed_nodes={'Cloud Region A'}, title='DIGITAL TWIN — CLOUD FAILURE PROPAGATION').show()
print('SCENARIO PROPAGATION GRAPH — only validated paths')
scenario_propagation_graph_figure(graph, cloud_failure_result.propagation_trace, failed_nodes={'Cloud Region A'}, title='CLOUD FAILURE — FOCUSED PROPAGATION').show()

FULL DIGITAL TWIN GRAPH — cloud failure impact state


SCENARIO PROPAGATION GRAPH — only validated paths


## CLOUD REGION FAILURE — DEPENDENCY AND IMPACT ANALYSIS
**Analysis scope: INDEPENDENT `cloud_region_a_8hr` RUN.** This is not combined with the Section 7 scenarios or the flagship combined stress.

Infrastructure-only input:

Region A fails at hour 0 for eight hours.

Region B activates after three hours at 70% capacity.

At hour 8 the primary returns, and configurable recovery capacity clears the accumulated backlog.

NetworkX derives the blast radius and SimPy derives the timeline and backlog.

In [ ]:
from digital_twin.config import load_scenario

# Purpose: This cell performs a detailed analysis of a specific 8-hour cloud region A failure scenario.
# It loads the scenario configuration, runs the simulation, and then presents the results in several parts:
# scenario parameters, blast radius, operational timeline, operational impact, financial impact, causal propagation paths,
# and a comparison with an alternative (faster backup activation) scenario.

# Input:
# - `cloud_region_a_8hr` (str): The name of the scenario to load and run.
# - `engine` (ScenarioEngine object): The initialized scenario engine from SECTION 5.
# - `graph` (NetworkX Graph object): The bank's dependency graph, initialized in SECTION 5.

# Interdependencies:
# - Relies on `load_scenario` from `digital_twin.config` to load scenario parameters.
# - Relies on the `engine` object from SECTION 5 to run the scenario.
# - Relies on the `graph` object from SECTION 5 for visualization of propagation.
# - Uses visualization functions (`scenario_propagation_graph_figure`, `cloud_outage_timeline_figure`)
#   and dependency analysis (`top_propagation_paths`) imported in SECTION 3.

# Load the configuration for the 'cloud_region_a_8hr' scenario.
cloud_8hr_config = load_scenario('cloud_region_a_8hr')
# Extract operational shock parameters from the loaded configuration.
cloud_8hr_params = cloud_8hr_config['shocks']['operational']

# Run the 'cloud_region_a_8hr' scenario through the engine.
cloud_8hr = engine.run('cloud_region_a_8hr')

# Extract operational and liquidity impact details from the scenario result.
cloud_8hr_operational = cloud_8hr.impacts['operational_impact']
cloud_8hr_liquidity = cloud_8hr.impacts['liquidity_impact']

print('1. SCENARIO')
# Output: Displays a DataFrame summarizing the key parameters of the 8-hour cloud failure scenario.
display(pd.DataFrame([
    {
        'Failed region': cloud_8hr_params['failed_region'],
        'Region A failure (hours)': cloud_8hr_params['outage_duration_hours'],
        'Backup region': cloud_8hr_params['backup_region'],
        'Region B activation (hours)': cloud_8hr_params['backup_activation_delay_hours'],
        'Backup capacity': cloud_8hr_params['backup_capacity_pct'],
        'Post-recovery capacity': cloud_8hr_params['post_recovery_capacity_pct'],
        'Simulation horizon': cloud_8hr_params['simulation_horizon_hours'],
    }
]))

print('2. DEPENDENCY / BLAST RADIUS')
# Output: Displays a DataFrame summarizing the types and counts of affected nodes (applications, services, customers, financial).
# Displays a focused propagation graph showing the actual dependency paths impacted by the cloud failure.
blast_radius_table = pd.DataFrame([
    {'Node Type': node_type, 'Affected Nodes': ', '.join(nodes), 'Count': len(nodes)}
    for node_type, nodes in {
        'Applications': cloud_8hr_operational['applications_affected'],
        'Business Services': cloud_8hr_operational['business_services_affected'],
        'Customer Segments': cloud_8hr_operational['customer_segments_affected'],
        'Financial / Risk Nodes': cloud_8hr_operational['financial_risk_nodes_affected'],
    }.items()
])
display(blast_radius_table)
scenario_propagation_graph_figure(graph, cloud_8hr.propagation_trace, failed_nodes={'Cloud Region A'}, title='CLOUD REGION A 8-HOUR FAILURE — ACTUAL DEPENDENCY PATHS').show()

print('3. TIMELINE')
# Output: Displays an interactive timeline visualization of the operational state, capacity, and backlog over time.
# Displays a DataFrame showing the detailed operational timeseries data (hour, operating state, capacity, backlog).
cloud_outage_timeline_figure(
    cloud_8hr.operational_timeseries, cloud_8hr_params['outage_duration_hours'],
    cloud_8hr_params['backup_activation_delay_hours'],
).show()
display(cloud_8hr.operational_timeseries[['hour','operating_state','capacity_fraction','backlog_bn','event']])

print('4. OPERATIONAL IMPACT')
# Output: Displays a DataFrame summarizing key operational metrics (applications affected, services affected, payment availability, backlog).
operational_impact_table = pd.DataFrame([
    {
        'Applications affected': len(cloud_8hr_operational['applications_affected']),
        'Services affected': len(cloud_8hr_operational['business_services_affected']),
        'Payment availability': cloud_8hr.metrics['payment_availability'],
        'Maximum backlog (USD bn)': cloud_8hr.metrics['payment_backlog_bn'],
        'Primary recovery (hours)': cloud_8hr_operational['primary_recovery_time_hours'],
        'Backlog clearance / total recovery (hours)': cloud_8hr_operational['backlog_clearance_time_hours'],
        'Customers affected': cloud_8hr.metrics['customers_affected'],
    }
])
display(operational_impact_table)

print('5. FINANCIAL IMPACT')
# Output: Displays a DataFrame summarizing key financial metrics (deposit outflow, liquidity consumed, LCR changes, CET1 changes).
financial_impact_table = pd.DataFrame([
    {
        'Deposit outflow (USD bn)': cloud_8hr.metrics['deposit_outflow_bn'],
        'Liquidity consumed (USD bn)': cloud_8hr.metrics['cash_consumed_bn'],
        'HQLA usage (USD bn)': cloud_8hr.metrics['hqla_consumed_bn'],
        'Emergency funding (USD bn)': cloud_8hr.metrics['emergency_funding_requirement_bn'],
        'Operational loss (USD bn)': cloud_8hr.metrics['operational_loss_bn'],
        'LCR before': cloud_8hr.baseline['lcr'], 'LCR after': cloud_8hr.metrics['lcr'],
        'CET1 before (USD bn)': cloud_8hr.baseline['cet1_capital_bn'],
        'CET1 after (USD bn)': cloud_8hr.metrics['cet1_capital_bn'],
    }
])
display(financial_impact_table)

print('6. CAUSAL PROPAGATION — model-generated, existing edges only')
# Output: Displays a DataFrame showing the top 5 most material causal propagation paths identified in the scenario.
cloud_8hr_paths = top_propagation_paths(cloud_8hr.propagation_trace, top_n=5)
display(pd.DataFrame([{'Path': path['path'], 'Material Effect': path['material_effect']} for path in cloud_8hr_paths]))

print('7. COMPARISON — identical scenario with Region B activation reduced to one hour')
# Output: Displays a DataFrame comparing key metrics of the original 3-hour backup activation scenario
# with a hypothetical 1-hour backup activation scenario, highlighting the changes.
cloud_1hr_backup = engine.run('cloud_region_a_8hr', overrides={'operational': {'backup_activation_delay_hours': 1}})
comparison_metrics = [
    ('Payment backlog (USD bn)', 'payment_backlog_bn'),
    ('Customers affected', 'customers_affected'),
    ('Deposit outflow (USD bn)', 'deposit_outflow_bn'),
    ('Operational loss (USD bn)', 'operational_loss_bn'),
    ('Liquidity consumed (USD bn)', 'cash_consumed_bn'),
    ('LCR', 'lcr'),
    ('Total recovery time (hours)', 'recovery_time_hours'),
]
cloud_backup_comparison = pd.DataFrame([
    {'Metric': label, '3-hour activation': cloud_8hr.metrics[key],
     '1-hour activation': cloud_1hr_backup.metrics[key],
     'Change': cloud_1hr_backup.metrics[key] - cloud_8hr.metrics[key]}
    for label, key in comparison_metrics
])
display(cloud_backup_comparison)

1. SCENARIO


,Failed region,Region A failure (hours),Backup region,Region B activation (hours),Backup capacity,Post-recovery capacity,Simulation horizon
0,Cloud Region A,8,Cloud Region B,3,0.7,1.25,24


2. DEPENDENCY / BLAST RADIUS


,Node Type,Affected Nodes,Count
0,Applications,"Cross-Border Payments, Domestic Payments, Iden...",5
1,Business Services,"Cross-Border Payments, Domestic Payments",2
2,Customer Segments,"Corporate, Private Banking, Retail, SME",4
3,Financial / Risk Nodes,"CET1 Ratio, Capital Position, Corporate Deposi...",6


3. TIMELINE


...

,Deposit outflow (USD bn),Liquidity consumed (USD bn),HQLA usage (USD bn),Emergency funding (USD bn),Operational loss (USD bn),LCR before,LCR after,CET1 before (USD bn),CET1 after (USD bn)
0,0.6324,0.6324,0.6324,0.0,0.104177,5.47619,4.696444,8.0,7.895823


6. CAUSAL PROPAGATION — model-generated, existing edges only


,Path,Material Effect
0,Cloud Region A → Domestic Payments → Corporate...,1.988100e+06
1,Retail → Deposit Outflows → Liquidity Position...,7.797469e-01
2,SME → Deposit Outflows → Liquidity Position → LCR,7.797469e-01
3,Corporate Deposits → Deposit Outflows → Liquid...,7.797469e-01
4,Private Banking → Deposit Outflows → Liquidity...,7.797469e-01


7. COMPARISON — identical scenario with Region B activation reduced to one hour


,Metric,3-hour activation,1-hour activation,Change
0,Payment backlog (USD bn),8.302826e-01,3.822826e-01,-0.448000
1,Customers affected,1.988100e+06,1.369580e+06,-618520.000000
2,Deposit outflow (USD bn),6.324000e-01,4.356533e-01,-0.196747
3,Operational loss (USD bn),1.041771e-01,9.158274e-02,-0.012594
4,Liquidity consumed (USD bn),6.324000e-01,4.356533e-01,-0.196747
5,LCR,4.696444e+00,4.918127e+00,0.221684
6,Total recovery time (hours),1.400000e+01,1.100000e+01,-3.000000


## CLOUD RESILIENCE DIGITAL TWIN
**Scope:** continues the same independent `cloud_region_a_8hr` result.

These deployment, failover, propagation, timeline, concentration, and cloud executive views are generated from `bank.applications`, `engine.graph`, and validated SimPy output.

In [ ]:
# Purpose: This cell provides a comprehensive set of visualizations and summaries related to the cloud region A failure scenario.
# It details the cloud deployment map, failover states at different hours, business impact graph, outage timeline,
# concentration risk analysis, and an executive impact summary.

# Input:
# - `bank.applications` (DataFrame): Information about the bank's applications, their primary and backup regions.
# - `cloud_8hr_operational` (dict): Operational impact details from the `cloud_8hr` scenario result, including application failover states.
# - `cloud_8hr.propagation_trace` (list): Detailed trace of how the failure propagated through the dependency graph.
# - `cloud_8hr.operational_timeseries` (DataFrame): Time-series data for operational metrics during the outage.
# - `cloud_8hr_params` (dict): Parameters of the cloud failure scenario, including outage and backup activation durations.
# - `graph` (NetworkX Graph object): The bank's dependency graph, initialized in SECTION 5.
# - `cloud_8hr.metrics` (dict): Key metrics calculated for the cloud failure scenario.
# - `cloud_8hr.baseline` (dict): Baseline metrics for comparison.

# Interdependencies:
# - Relies on the `bank` object from SECTION 4 for application data.
# - Relies on the `cloud_8hr` scenario result and its extracted components (`cloud_8hr_operational`, `cloud_8hr.propagation_trace`,
#   `cloud_8hr.operational_timeseries`, `cloud_8hr.metrics`, `cloud_8hr.baseline`) from the previous cell (`fab48822`).
# - Uses visualization functions (`cloud_deployment_map_figure`, `scenario_propagation_graph_figure`,
#   `cloud_outage_timeline_figure`, `cloud_concentration_figure`) and analysis functions (`cloud_concentration_metrics`)
#   imported from `digital_twin.visualizations` and `digital_twin.dependency_graph` (in SECTION 3).

# Prepare DataFrames for application failover states at hour 0 and backup activation time.
cloud_states_h0 = pd.DataFrame(cloud_8hr_operational['application_failover_states']['hour_0'])
cloud_states_h3 = pd.DataFrame(cloud_8hr_operational['application_failover_states']['backup_activation'])

print('VIEW 1 — CLOUD DEPLOYMENT MAP')
# Output: Displays a visualization of the overall cloud deployment strategy, showing where applications are hosted.
# Displays a DataFrame with details about each application's primary region, backup region, mode, and criticality.
cloud_deployment_map_figure(bank.applications, title='CLOUD DEPLOYMENT MAP — PRIMARY AND BACKUP PLACEMENT').show()
display(bank.applications[['application','primary_region','backup_region','backup_mode','failover_time_minutes','normal_capacity_pct','backup_capacity_pct','criticality']])

print('VIEW 2 — REGION A FAILURE AT HOUR 0')
# Output: Displays a visualization of the cloud deployment map at the moment Region A fails, highlighting affected applications.
# Displays a DataFrame showing the status and effective capacity of applications directly affected by the failure at hour 0.
cloud_deployment_map_figure(bank.applications, cloud_states_h0, title='HOUR 0 — CLOUD REGION A FAILED').show()
display(cloud_states_h0.loc[cloud_states_h0['affected_by_failure'], ['application','primary_region','backup_region','has_designated_backup','application_status','effective_capacity_pct']])

print('VIEW 3 — HOUR 3 FAILOVER')
# Output: Displays a visualization of the cloud deployment map after 3 hours, showing the state after backup region B activates.
# Displays a DataFrame detailing the status, active region, and effective capacity of affected applications after failover.
cloud_deployment_map_figure(bank.applications, cloud_states_h3, title='HOUR 3 — REGION B ACTIVE AT 70% CAPACITY').show()
display(cloud_states_h3.loc[cloud_states_h3['affected_by_failure'], ['application','directly_hosted_in_failed_region','application_status','active_region','effective_capacity_pct']])

print('VIEW 4 — BUSINESS IMPACT GRAPH')
# Output: Displays a focused propagation graph illustrating how the Region A failure impacts business services, customers, and financial nodes.
scenario_propagation_graph_figure(graph, cloud_8hr.propagation_trace, failed_nodes={'Cloud Region A'}, title='REGION A FAILURE — BUSINESS, CUSTOMER, AND FINANCIAL PROPAGATION').show()

print('VIEW 5 — OUTAGE TIMELINE')
# Output: Displays an interactive timeline visualization of the operational state, capacity, and backlog during the outage and recovery.
cloud_outage_timeline_figure(cloud_8hr.operational_timeseries, cloud_8hr_params['outage_duration_hours'], cloud_8hr_params['backup_activation_delay_hours']).show()

print('VIEW 6 — CONCENTRATION RISK')
# Output: Displays a visualization of cloud concentration risk metrics.
# Displays a DataFrame summarizing various concentration risk measures, such as applications per region, critical applications,
# and single points of failure related to cloud regions.
cloud_concentration = cloud_concentration_metrics(bank, graph)
cloud_concentration_figure(cloud_concentration).show()
cloud_concentration_table = pd.DataFrame([
    {'Measure': 'Applications in Cloud Region A', 'Value': ', '.join(cloud_concentration['applications_by_region']['Cloud Region A'])},
    {'Measure': 'Applications in Cloud Region B', 'Value': ', '.join(cloud_concentration['applications_by_region']['Cloud Region B'])},
    {'Measure': 'Critical applications in Region A', 'Value': ', '.join(cloud_concentration['critical_applications_by_region']['Cloud Region A'])},
    {'Measure': 'Critical applications in Region B', 'Value': ', '.join(cloud_concentration['critical_applications_by_region']['Cloud Region B'])},
    {'Measure': 'Applications without backup', 'Value': ', '.join(cloud_concentration['applications_without_backup'])},
    {'Measure': 'Applications solely dependent on Region A', 'Value': ', '.join(cloud_concentration['applications_solely_region_a'])},
    {'Measure': 'Business services solely dependent on Region A', 'Value': ', '.join(cloud_concentration['business_services_solely_region_a'])},
    {'Measure': 'Customer segments exposed to Region A', 'Value': ', '.join(cloud_concentration['customer_segments_exposed_to_region_a'])},
    {'Measure': 'Important single points of failure', 'Value': ', '.join(cloud_concentration['single_points_of_failure'])},
])
display(cloud_concentration_table)

print('VIEW 7 — EXECUTIVE IMPACT SUMMARY')
# Output: Displays a concise executive summary DataFrame of the cloud failure's impact across technology, operations, customers, and financial categories.
affected_state_rows = cloud_states_h3.loc[cloud_states_h3['affected_by_failure']]
direct_region_a = affected_state_rows.loc[affected_state_rows['directly_hosted_in_failed_region']]
recovered_region_b = direct_region_a.loc[direct_region_a['active_region'] == 'Cloud Region B', 'application'].tolist()
no_backup_apps = direct_region_a.loc[~direct_region_a['has_designated_backup'], 'application'].tolist()
cloud_executive_summary = pd.DataFrame([
    {'Category': 'TECHNOLOGY', 'Measure': 'Applications affected', 'Value': len(cloud_8hr_operational['applications_affected']), 'Detail': ', '.join(cloud_8hr_operational['applications_affected'])},
    {'Category': 'TECHNOLOGY', 'Measure': 'Recovered through Region B', 'Value': len(recovered_region_b), 'Detail': ', '.join(recovered_region_b)},
    {'Category': 'TECHNOLOGY', 'Measure': 'Without Region B backup', 'Value': len(no_backup_apps), 'Detail': ', '.join(no_backup_apps)},
    {'Category': 'OPERATIONS', 'Measure': 'Business services affected', 'Value': len(cloud_8hr_operational['business_services_affected']), 'Detail': ', '.join(cloud_8hr_operational['business_services_affected'])},
    {'Category': 'OPERATIONS', 'Measure': 'Maximum payment backlog (USD bn)', 'Value': cloud_8hr.metrics['payment_backlog_bn'], 'Detail': ''},
    {'Category': 'OPERATIONS', 'Measure': 'Total recovery time (hours)', 'Value': cloud_8hr.metrics['recovery_time_hours'], 'Detail': ''},
    {'Category': 'CUSTOMERS', 'Measure': 'Customers affected', 'Value': cloud_8hr.metrics['customers_affected'], 'Detail': ''},
    {'Category': 'CUSTOMERS', 'Measure': 'Affected segments', 'Value': len(cloud_8hr_operational['customer_segments_affected']), 'Detail': ', '.join(cloud_8hr_operational['customer_segments_affected'])},
    {'Category': 'FINANCIAL', 'Measure': 'Deposit outflow (USD bn)', 'Value': cloud_8hr.metrics['deposit_outflow_bn'], 'Detail': ''},
    {'Category': 'FINANCIAL', 'Measure': 'Liquidity consumed (USD bn)', 'Value': cloud_8hr.metrics['cash_consumed_bn'], 'Detail': ''},
    {'Category': 'FINANCIAL', 'Measure': 'HQLA usage (USD bn)', 'Value': cloud_8hr.metrics['hqla_consumed_bn'], 'Detail': ''},
    {'Category': 'FINANCIAL', 'Measure': 'LCR baseline → stressed', 'Value': cloud_8hr.metrics['lcr'], 'Detail': f"{cloud_8hr.baseline['lcr']:.3f} → {cloud_8hr.metrics['lcr']:.3f}"},
    {'Category': 'FINANCIAL', 'Measure': 'Operational loss (USD bn)', 'Value': cloud_8hr.metrics['operational_loss_bn'], 'Detail': ''},
])
display(cloud_executive_summary)

VIEW 1 — CLOUD DEPLOYMENT MAP


,application,primary_region,backup_region,backup_mode,failover_time_minutes,normal_capacity_pct,backup_capacity_pct,criticality
0,Core Banking,Primary Data Centre,Backup Data Centre,active-passive,60.0,100,100,Critical
1,Domestic Payments,Cloud Region A,Cloud Region B,warm standby,180.0,100,70,Critical
2,Cross-Border Payments,Cloud Region A,None,none,NaN,100,0,Critical
3,Mobile Banking,Primary Data Centre,Backup Data Centre,active-passive,30.0,100,100,High
4,Internet Banking,Primary Data Centre,Backup Data Centre,active-passive,30.0,100,100,High
5,Treasury Platform,Primary Data Centre,Backup Data Centre,warm standby,60.0,100,90,Critical
6,Identity Service,Cloud Region A,Cloud Region B,hot standby,30.0,100,90,Critical
7,Fraud Monitoring,Primary Data Centre,Backup Data Centre,active-passive,30.0,100,100,High
8,Credit Risk Engine,Primary Data Centre,Backup Data Centre,warm standby,120.0,100,80,High
9,Liquidity Risk Platform,Primary Data Centre,Backup Data Centre,warm standby,120.0,100,80,High


VIEW 2 — REGION A FAILURE AT HOUR 0


,application,primary_region,backup_region,has_designated_backup,application_status,effective_capacity_pct
1,Domestic Payments,Cloud Region A,Cloud Region B,True,Waiting for failover,0.0
2,Cross-Border Payments,Cloud Region A,None,False,No backup / unavailable,0.0
3,Mobile Banking,Primary Data Centre,Backup Data Centre,False,Dependency impacted,0.0
4,Internet Banking,Primary Data Centre,Backup Data Centre,False,Dependency impacted,0.0
6,Identity Service,Cloud Region A,Cloud Region B,True,Waiting for failover,0.0


VIEW 3 — HOUR 3 FAILOVER


...

VIEW 5 — OUTAGE TIMELINE


VIEW 6 — CONCENTRATION RISK


,Measure,Value
0,Applications in Cloud Region A,"Cross-Border Payments, Domestic Payments, Iden..."
1,Applications in Cloud Region B,"Domestic Payments, Identity Service"
2,Critical applications in Region A,"Cross-Border Payments, Domestic Payments, Iden..."
3,Critical applications in Region B,"Domestic Payments, Identity Service"
4,Applications without backup,Cross-Border Payments
5,Applications solely dependent on Region A,Cross-Border Payments
6,Business services solely dependent on Region A,Cross-Border Payments
7,Customer segments exposed to Region A,"Corporate, Private Banking, Retail, SME"
8,Important single points of failure,"Capital Position, Cloud Region A, Core Banking..."


VIEW 7 — EXECUTIVE IMPACT SUMMARY


,Category,Measure,Value,Detail
0,TECHNOLOGY,Applications affected,5.000000e+00,"Cross-Border Payments, Domestic Payments, Iden..."
1,TECHNOLOGY,Recovered through Region B,2.000000e+00,"Domestic Payments, Identity Service"
2,TECHNOLOGY,Without Region B backup,1.000000e+00,Cross-Border Payments
3,OPERATIONS,Business services affected,2.000000e+00,"Cross-Border Payments, Domestic Payments"
4,OPERATIONS,Maximum payment backlog (USD bn),8.302826e-01,
5,OPERATIONS,Total recovery time (hours),1.400000e+01,
6,CUSTOMERS,Customers affected,1.988100e+06,
7,CUSTOMERS,Affected segments,4.000000e+00,"Corporate, Private Banking, Retail, SME"
8,FINANCIAL,Deposit outflow (USD bn),6.324000e-01,
9,FINANCIAL,Liquidity consumed (USD bn),6.324000e-01,


## SECTION 8 — FLAGSHIP COMBINED STRESS
**Analysis scope: FLAGSHIP `combined_stress` RUN.** This is a new, independent engine run; it does not automatically combine the preceding Section 7 result objects or reuse the dedicated eight-hour cloud configuration.

Its own YAML simultaneously specifies FX, volatility, deposit-withdrawal, counterparty-credit, and shorter cloud/payment impairment shocks. These propagate through graph edges and financial rules into market loss, payment disruption, behavioural withdrawals, liquidity, LCR, loss, and CET1. The resulting `combined` object is the principal input to Sections 9–13.

In [ ]:
# Purpose: This cell executes the 'combined_stress' scenario, which represents a complex, multi-faceted stress event impacting the bank simultaneously.
# It involves various shocks (FX, volatility, deposit-withdrawal, counterparty-credit, and cloud/payment impairment) propagating through the bank's system.
# This is the flagship simulation for the analysis in subsequent sections.

# Input:
# - `engine` (ScenarioEngine object): The initialized scenario engine from SECTION 5.
# - 'combined_stress' (str): The name of the scenario configuration to load and run.
# - `graph` (NetworkX Graph object): The bank's dependency graph, initialized in SECTION 5, used for visualization.

# Interdependencies:
# - Relies on the `engine` object from SECTION 5 to perform the simulation.
# - Relies on the `graph` object from SECTION 5 for visualizing propagation paths.
# - Uses visualization functions (`interactive_dependency_graph`, `scenario_propagation_graph_figure`,
#   `baseline_vs_stressed`, `payment_backlog_over_time`) and analysis functions (`top_propagation_paths`)
#   imported in SECTION 3.
# - The output `combined` object serves as the primary input for SECTION 9, 10, 11, 12, 13, and 15.

# Run the 'combined_stress' scenario through the engine.
combined = engine.run('combined_stress')

# Output:
# - `combined` (ScenarioResult object): A comprehensive object containing all metrics, impacts, and propagation traces
#   from the combined stress scenario.
# - Displays a DataFrame summarizing the key metrics from the `combined` scenario result.
# - Prints the top propagation paths identified during the simulation.
# - Prints any risk-limit breaches observed in the scenario.
# - Displays a DataFrame showing the top 5 most material causal propagation paths.
# - Displays an interactive dependency graph highlighting the impact of the combined stress.
# - Displays a focused propagation graph for the combined stress, showing only affected paths.
# - Displays a comparison visualization of baseline versus stressed metrics.
# - Displays a timeline visualization of payment backlog over time during the combined stress.
display(pd.DataFrame([combined.metrics]))
print('Propagation paths:')
for path in combined.propagation_paths: print(' → '.join(path))
print('Risk-limit breaches:', combined.risk_limit_breaches)
combined_top_paths = top_propagation_paths(combined.propagation_trace, top_n=5)
display(pd.DataFrame([{'Path': p['path'], 'Material Effect': p['material_effect']} for p in combined_top_paths]))
interactive_dependency_graph(graph, combined.propagation_trace, title='DIGITAL TWIN — COMBINED STRESS IMPACT MAP').show()
scenario_propagation_graph_figure(graph, combined.propagation_trace, title='COMBINED STRESS — FOCUSED PROPAGATION').show()
baseline_vs_stressed(combined.baseline, combined.metrics).show()
payment_backlog_over_time(combined.operational_timeseries).show()

,total_estimated_loss_bn,market_loss_bn,credit_loss_bn,operational_loss_bn,liquidity_pnl_loss_bn,cash_position_bn,hqla_bn,lcr,cet1_capital_bn,risk_weighted_assets_bn,...,customers_affected,applications_affected,recovery_time_hours,deposit_outflow_bn,cash_consumed_bn,hqla_consumed_bn,emergency_funding_requirement_bn,funding_cost_bn,asset_liquidation_bn,realised_asset_sale_loss_bn
0,0.816617,0.48,0.281286,0.054334,0.000997,-0.242533,15.3,1.029434,7.183383,51.84,...,1104499,5,12.0,10.223333,10.242533,10.0,0.242533,0.000997,0.0,0.0


Propagation paths:
Cloud Region A → Identity Service → Mobile Banking → Retail → Deposit Outflows → Liquidity Position
Cloud Region A → Identity Service → Internet Banking → Retail → Deposit Outflows → Liquidity Position
Cloud Region A → Identity Service → Internet Banking → SME → Deposit Outflows → Liquidity Position
Cloud Region A → Domestic Payments → SME → Deposit Outflows → Liquidity Position
Cloud Region A → Domestic Payments → Corporate → Corporate Deposits → Deposit Outflows → Liquidity Position
Cloud Region A → Cross-Border Payments → Corporate → Corporate Deposits → Deposit Outflows → Liquidity Position
Cloud Region A → Identity Service → Mobile Banking → Retail → Deposit Outflows → Liquidity Position → Capital Position
Cloud Region A → Identity Service → Internet Banking → Retail → Deposit Outflows → Liquidity Position → Capital Position
Cloud Region A → Identity Service → Internet Banking → SME → Deposit Outflows → Liquidity Position → Capital Position
Cloud Region A → Dome

,Path,Material Effect
0,Cloud Region A → Domestic Payments → Corporate...,1.104499e+06
1,USD Shock → FX Exposure → Market P&L → Capital...,8.166174e-01
2,Volatility Shock → Market P&L → Capital Positi...,8.166174e-01
3,Major Counterparty → Credit Loss → Capital Pos...,8.166174e-01
4,Retail → Deposit Outflows → Liquidity Position...,1.024253e+01


## LCR VALIDATION AND LIQUIDITY BRIDGE
**Scope:** deterministic flagship `combined_stress` from Section 8 versus its baseline.

This validates `LCR = eligible HQLA / stressed 30-day net cash outflows`; it is not a regulatory LCR calculation. The bridge shows numerator, denominator, assumptions, and reconciliation checks using actual Python values.

In [ ]:
# Purpose: This cell performs a detailed validation of the Liquidity Coverage Ratio (LCR) calculation for the 'combined_stress' scenario.
# It breaks down the LCR into its components (HQLA and net cash outflows) and presents a 'bridge' to show how these values change
# from baseline to stressed conditions. It also includes validation checks to ensure mathematical and conceptual consistency.

# Input:
# - `combined.impacts['lcr_bridge']` (dict): Detailed breakdown of LCR components under baseline and stressed conditions.
# - `combined.impacts['lcr_validation_checks']` (dict): Results of internal consistency checks for LCR calculation.
# - `combined.baseline['lcr']` (float): Baseline LCR value from the `combined` scenario result.
# - `combined.metrics['lcr']` (float): Stressed LCR value from the `combined` scenario result.

# Interdependencies:
# - Relies entirely on the `combined` ScenarioResult object generated in SECTION 8, particularly its `impacts`, `baseline`, and `metrics` attributes.

# Extract the LCR bridge and validation checks from the combined scenario results.
lcr_bridge = pd.DataFrame(combined.impacts['lcr_bridge'])
lcr_validation_checks = pd.DataFrame(combined.impacts['lcr_validation_checks'])

# Output:
# - Prints the Baseline and Stressed Prototype LCR values.
# - Displays the 'HQLA bridge' DataFrame, showing changes in High Quality Liquid Assets.
# - Displays the 'Cash-outflow bridge' DataFrame, showing changes in net cash outflows.
# - Displays the full 'LCR bridge' DataFrame, summarizing all components.
# - Prints key assumptions underlying the LCR calculation.
# - Displays a DataFrame of `lcr_validation_checks`, indicating whether internal consistency checks passed.
# - Asserts that all validation checks passed; if not, an error will be raised.
# - Prints a classification message regarding the consistency of the LCR calculation.
print(f"Baseline prototype LCR: {combined.baseline['lcr']:.6f}")
print(f"Stressed prototype LCR: {combined.metrics['lcr']:.6f}")
print('HQLA bridge')
display(lcr_bridge.iloc[:6])
print('Cash-outflow bridge')
display(lcr_bridge.iloc[6:16])
print('LCR bridge')
display(lcr_bridge)
print('Key assumptions: 15% securities haircut; USD billions; fixed $2bn eligible inflow cap; no wholesale, credit-line or other outflow components are modelled.')
display(lcr_validation_checks)
assert lcr_validation_checks['Passed'].all()
print('Classification: A — mathematically and conceptually consistent with the stated synthetic prototype assumptions; the high baseline ratio reflects those assumptions, not a regulatory calibration.')

Baseline prototype LCR: 5.476190
Stressed prototype LCR: 1.029434
HQLA bridge


,Component,Baseline,Stress Change,Stressed,Explanation
0,Starting cash / eligible liquid assets,10.0,-10.0,0.0,Cash is floored at zero in eligible HQLA.
1,Securities before haircut,18.0,0.0,18.0,Gross synthetic securities stock.
2,Securities haircut,2.7,0.0,2.7,Configured haircut deducted once.
3,Eligible securities,15.3,0.0,15.3,Securities after configured haircut.
4,Other eligible HQLA,0.0,0.0,0.0,Not modelled; zero.
5,Total eligible HQLA,25.3,-10.0,15.3,Eligible cash plus eligible securities and oth...


Cash-outflow bridge


,Component,Baseline,Stress Change,Stressed,Explanation
6,Retail cash outflow,2.10,2.450000,4.550000,Baseline 30-day flow plus scenario increment.
7,SME cash outflow,1.20,1.490000,2.690000,Baseline 30-day flow plus scenario increment.
8,Corporate cash outflow,3.00,5.789583,8.789583,Baseline 30-day flow plus scenario increment.
9,Private banking cash outflow,0.32,0.493750,0.813750,Baseline 30-day flow plus scenario increment.
10,Wholesale funding outflow,0.00,0.000000,0.000000,Not modelled; zero.
11,Credit-line drawdown,0.00,0.000000,0.000000,Not modelled; zero.
12,Margin/collateral requirement,0.00,0.019200,0.019200,Market-loss-linked margin call.
13,Other stressed outflows,0.00,0.000000,0.000000,Not modelled; zero.
14,Eligible inflows,2.00,0.000000,2.000000,"Configured fixed inflow cap, deducted from gro..."
15,Net stressed 30-day cash outflow,4.62,10.242533,14.862533,Gross modelled outflows less eligible inflows.


LCR bridge


,Component,Baseline,Stress Change,Stressed,Explanation
0,Starting cash / eligible liquid assets,10.00000,-10.000000,0.000000,Cash is floored at zero in eligible HQLA.
1,Securities before haircut,18.00000,0.000000,18.000000,Gross synthetic securities stock.
2,Securities haircut,2.70000,0.000000,2.700000,Configured haircut deducted once.
3,Eligible securities,15.30000,0.000000,15.300000,Securities after configured haircut.
4,Other eligible HQLA,0.00000,0.000000,0.000000,Not modelled; zero.
5,Total eligible HQLA,25.30000,-10.000000,15.300000,Eligible cash plus eligible securities and oth...
6,Retail cash outflow,2.10000,2.450000,4.550000,Baseline 30-day flow plus scenario increment.
7,SME cash outflow,1.20000,1.490000,2.690000,Baseline 30-day flow plus scenario increment.
8,Corporate cash outflow,3.00000,5.789583,8.789583,Baseline 30-day flow plus scenario increment.
9,Private banking cash outflow,0.32000,0.493750,0.813750,Baseline 30-day flow plus scenario increment.


Key assumptions: 15% securities haircut; USD billions; fixed $2bn eligible inflow cap; no wholesale, credit-line or other outflow components are modelled.


,Validation Check,Passed,Observed,Expected
0,Baseline LCR = HQLA / net outflow,True,5.47619,5.47619
1,Stressed LCR = HQLA / net outflow,True,1.029434,1.029434
2,Baseline HQLA reconciles,True,25.3,25.3
3,Stressed HQLA reconciles,True,15.3,15.3
4,HQLA is non-negative,True,15.3,>= 0
5,Net stressed outflow is positive,True,14.862533,> 0
6,USD billion units are explicit,True,USD bn,USD bn


Classification: A — mathematically and conceptually consistent with the stated synthetic prototype assumptions; the high baseline ratio reflects those assumptions, not a regulatory calibration.


## SECTION 9 — Monte Carlo simulation
**Purpose:** This section performs a Monte Carlo simulation to assess the banking institution's resilience under 1,000 stochastic variations of the `combined_stress` scenario.
This approach accounts for uncertainties in certain parameters (e.g., recovery time), providing a probabilistic view of potential outcomes beyond a single deterministic run.



**Analysis scope: 1,000 STOCHASTIC VARIATIONS OF `combined_stress`.** This is not a Monte Carlo analysis of the Section 7 scenario set or the independent eight-hour cloud scenario.

Python varies selected combined-stress parameters, reruns the full engine, and reports percentiles and breach probabilities. Fixed seed 42 makes the experiment reproducible. One row in `mc_results` represents one combined-stress simulation.

In [ ]:
# Purpose: This cell executes the Monte Carlo simulation for the 'combined_stress' scenario,
# generating 1,000 stochastic variations to assess the probabilistic outcomes of various metrics
# and the likelihood of breaching predefined risk limits.

# Input:
# - `engine` (ScenarioEngine object): The initialized scenario engine from SECTION 5.
# - `'combined_stress'` (str): The name of the scenario configuration to run.
# - `runs` (int): The number of Monte Carlo iterations (1000 in this case).
# - `seed` (int): A fixed random seed (42) for reproducibility.
# - `bank.risk_limits` (dict): The bank's risk limits, used to calculate breach probabilities.

# Interdependencies:
# - Relies on the `engine` object from SECTION 5.
# - Uses the `combined_stress` scenario definition from SECTION 8.
# - Uses `bank.risk_limits` defined during the bank setup.
# - Utilizes functions from `digital_twin.monte_carlo` (e.g., `run_monte_carlo`, `summarize_monte_carlo`,
#   `metric_percentiles`, `breach_probability_table`, `explain_probability_extremes`) and visualization
#   functions (`breach_probability_figure`, `loss_distribution`, `lcr_distribution`, `distribution_figure`)
#   imported in SECTION 3.

# Run the Monte Carlo simulation.
mc_results = run_monte_carlo(engine, 'combined_stress', runs=1000, seed=42)

# Summarize the Monte Carlo results, calculate metric percentiles, and breach probabilities.
mc_summary = summarize_monte_carlo(mc_results)
mc_percentiles = metric_percentiles(mc_results)
mc_breach_probabilities = breach_probability_table(mc_results, bank.risk_limits)
mc_extreme_diagnostics = explain_probability_extremes(mc_results, mc_breach_probabilities)

# Output:
# - `mc_results` (DataFrame): A DataFrame containing the results of each of the 1,000 Monte Carlo runs.
# - `mc_summary` (DataFrame): A statistical summary (mean, median, percentiles) of the Monte Carlo results.
# - `mc_percentiles` (DataFrame): Key percentiles (P5, Median, P95) for each risk metric.
# - `mc_breach_probabilities` (DataFrame): The probability of breaching each risk limit.
# - `mc_extreme_diagnostics` (DataFrame): Explanations for extreme probability outcomes.
# - Displays all these summary DataFrames.
# - Displays a `breach_probability_figure` visualization showing the probabilities of breaching risk limits.
# - Prints the overall probability of any risk-limit breach.
# - Displays `loss_distribution`, `lcr_distribution`, and `recovery_time_hours` distribution figures,
#   visualizing the spread of outcomes for these critical metrics.
display(mc_summary)
display(mc_percentiles)
display(mc_breach_probabilities)
display(mc_extreme_diagnostics)
breach_probability_figure(mc_breach_probabilities).show()
print('Probability of any risk-limit breach:', mc_summary.attrs['probability_risk_limit_breach'])
loss_distribution(mc_results).show(); lcr_distribution(mc_results).show()
distribution_figure(mc_results, 'recovery_time_hours', 'Recovery-time Distribution').show()

,metric,mean,median,p05,p95
0,total_loss_bn,8.276981e-01,8.290463e-01,0.657509,9.947924e-01
1,lowest_cash_bn,-2.580625e-01,-2.810460e-01,-2.262906,1.767948e+00
2,lcr,1.064637e+00,1.026774e+00,0.906242,1.328033e+00
3,cet1_ratio,1.383546e-01,1.383286e-01,0.135131,1.416376e-01
4,payment_backlog_bn,5.255752e-01,4.887560e-01,0.469820,6.355911e-01
5,payment_availability,8.876958e-01,8.958333e-01,0.837500,9.250000e-01
6,recovery_time_hours,1.270900e+01,1.200000e+01,8.000000,1.800000e+01
7,customers_affected,1.190783e+06,1.104499e+06,795240.000000,1.723019e+06


,Risk Metric,P5,Median,P95
0,total_loss_bn,0.657509,8.290463e-01,9.947924e-01
1,lowest_cash_bn,-2.262906,-2.810460e-01,1.767948e+00
2,lcr,0.906242,1.026774e+00,1.328033e+00
3,cet1_ratio,0.135131,1.383286e-01,1.416376e-01
4,payment_availability,0.837500,8.958333e-01,9.250000e-01
5,payment_backlog_bn,0.469820,4.887560e-01,6.355911e-01
6,recovery_time_hours,8.000000,1.200000e+01,1.800000e+01
7,customers_affected,795240.000000,1.104499e+06,1.723019e+06


,Risk Metric,Threshold,Probability of Breach,Severity,Breach Count,Simulation Runs
0,LCR warning,< 1.1,0.701,warning,701,1000
1,LCR critical,< 1,0.376,critical,376,1000
2,Negative cash,< 0,0.601,critical,601,1000
3,CET1 warning,< 0.12,0.000,warning,0,1000
4,CET1 critical,< 0.105,0.000,critical,0,1000
5,Payment availability warning,< 0.98,1.000,warning,1000,1000
6,Payment availability critical,< 0.95,1.000,critical,1000,1000
7,Severe total loss,> 2,0.000,severe,0,1000
8,Recovery-time threshold,> 4,1.000,warning,1000,1000


,Risk Metric,Probability of Breach,Observed Minimum,Observed Maximum,Unique Outcomes,Explanation
0,CET1 warning,0.0,0.133238,0.146061,1000,threshold is not exceeded by any sampled outcome
1,CET1 critical,0.0,0.133238,0.146061,1000,threshold is not exceeded by any sampled outcome
2,Payment availability warning,1.0,0.779167,0.925000,11,threshold is exceeded by every sampled outcome
3,Payment availability critical,1.0,0.779167,0.925000,11,threshold is exceeded by every sampled outcome
4,Severe total loss,0.0,0.428207,1.092959,1000,threshold is not exceeded by any sampled outcome
5,Recovery-time threshold,1.0,8.000000,22.000000,9,threshold is exceeded by every sampled outcome


Probability of any risk-limit breach: 1.0


## MONTE CARLO OPERATIONAL BREACH VALIDATION
**Purpose:** This section validates the results of the Monte Carlo simulation from Section 9, specifically focusing on operational breaches related to payment availability and recovery time.


**Analysis scope:** validates the Section 9 combined-stress Monte Carlo results. It diagnoses payment-availability and recovery-time breach probabilities, confirms that sampled recovery duration reaches the operational simulator, and explains 100% outcomes without changing thresholds or distributions.

In [ ]:
# Purpose: This cell performs detailed diagnostics for operational breaches identified in the Monte Carlo simulation.
# It specifically focuses on payment availability and recovery time metrics, providing insights into why their breach probabilities might be 100% or 0%.

# Input:
# - `mc_results` (DataFrame): The DataFrame containing the results of all 1,000 Monte Carlo runs from SECTION 9.
# - `bank.risk_limits` (dict): The bank's predefined risk limits, used to evaluate breaches.

# Interdependencies:
# - Relies on `mc_results` and `bank.risk_limits` which are outputs from SECTION 9.
# - Uses the `operational_breach_diagnostics` function and `threshold_distribution_figure` visualization function,
#   both imported from `digital_twin.monte_carlo` and `digital_twin.visualizations` respectively, in SECTION 3.

mc_operational_diagnostics = operational_breach_diagnostics(mc_results, bank.risk_limits)
display(mc_operational_diagnostics)
threshold_distribution_figure(mc_results, 'payment_availability', {'warning': bank.risk_limits['payment_availability']['warning'], 'critical': bank.risk_limits['payment_availability']['critical']}, 'Payment Availability — Monte Carlo and Limits').show()
threshold_distribution_figure(mc_results, 'recovery_time_hours', {'warning': bank.risk_limits['recovery_time_hours']['warning'], 'critical': bank.risk_limits['recovery_time_hours']['critical']}, 'Recovery Time — Monte Carlo and Limits').show()
print('Recovery duration is sampled and propagates to both metrics. Backup activation and capacity remain deterministic in this Monte Carlo design.')
print('All sampled outcomes breach because availability never reaches 95%, and recovery time never falls to 4 hours or below.')

# Output:
# - `mc_operational_diagnostics` (DataFrame): A DataFrame providing diagnostic details for operational breaches,
#   explaining the reasons for 0% or 100% breach probabilities.
# - Displays the `mc_operational_diagnostics` DataFrame.
# - Displays two `threshold_distribution_figure` visualizations:
#   - One for 'Payment Availability', showing its distribution relative to warning and critical thresholds.
#   - One for 'Recovery Time', showing its distribution relative to warning and critical thresholds.
# - Prints textual explanations for the observed breach patterns, specifically regarding why all sampled outcomes breach
#   for certain operational metrics.

,Metric,Threshold,Min,P5,Median,P95,Max,Breach Probability,Primary Driver,Deterministic or Stochastic?,Unique Outcomes,Classification
0,Payment availability warning,< 0.98,0.779167,0.8375,0.895833,0.925,0.925,1.0,"Sampled recovery duration; fixed outage start,...",Stochastic,11,B. Correct because stochastic range never cros...
1,Payment availability critical,< 0.95,0.779167,0.8375,0.895833,0.925,0.925,1.0,"Sampled recovery duration; fixed outage start,...",Stochastic,11,B. Correct because stochastic range never cros...
2,Recovery-time threshold,> 4,8.000000,8.0000,12.000000,18.000,22.000,1.0,Sampled recovery duration plus simulated backl...,Stochastic,9,B. Correct because stochastic range never cros...


Recovery duration is sampled and propagates to both metrics. Backup activation and capacity remain deterministic in this Monte Carlo design.
All sampled outcomes breach because availability never reaches 95%, and recovery time never falls to 4 hours or below.


## SECTION 10 — Management actions
**Purpose:** This section explores the impact of various management actions in response to the `combined_stress` scenario, which was run in SECTION 8. It evaluates how different individual and combined management strategies can mitigate the adverse effects of the stress.

**Analysis scope:** The analysis focuses on management responses to the `combined_stress` scenario. Each action (or combination of actions) is re-run against the same flagship combined stress. These results are specific to the combined stress and do not automatically apply to other independent scenarios, such as the eight-hour cloud scenario.


The section compares no action, every supported single action, and simultaneous multi-action responses across loss, liquidity, capital, operations, customers, and recovery.

In [ ]:
# Purpose: This cell orchestrates the analysis of various management actions in response to the 'combined_stress' scenario.
# It calculates the impact of individual actions, combinations of actions, and provides a baseline for comparison (no action).

# Input:
# - `engine` (ScenarioEngine object): The initialized scenario engine from SECTION 5.
# - `'combined_stress'` (str): The name of the scenario configuration, executed in SECTION 8.

# Interdependencies:
# - Relies on the `engine` object from SECTION 5.
# - Uses the `combined_stress` scenario definition and its results (`combined` object) from SECTION 8.
# - Utilizes functions from `digital_twin.action_engine` (e.g., `analyze_management_strategies`, `compare_actions`, `attribute_management_actions`)
#   imported in SECTION 3.
# - The outputs from this cell (`management_strategies`, `action_runs`, `before_action`, `after_action`, `management_action_attribution`)
#   are critical inputs for the 'MANAGEMENT RESPONSE DECISION LAB' (subsequent cells in SECTION 10), and SECTION 11, 12 and 15.

management_strategies = analyze_management_strategies(engine, 'combined_stress')
action_runs = management_strategies.individual_results
combined_actions = management_strategies.selected_combined_actions
before_action = management_strategies.no_action
after_action = management_strategies.selected_combined_result
_, _, action_comparison = compare_actions(engine, 'combined_stress', combined_actions)
management_action_attribution, management_action_traces = attribute_management_actions(engine, 'combined_stress')
print('Management strategy calculations completed. Detailed executive presentation follows below.')

# Output:
# - `management_strategies` (ManagementStrategyAnalysis object): A comprehensive object encapsulating all results
#   of the management strategy analysis (individual action results, combined results, comparisons, etc.).
# - `action_runs` (dict): A dictionary mapping management action names to their `ScenarioResult` objects.
# - `combined_actions` (list): A list of strings representing the names of management actions that were combined.
# - `before_action` (ScenarioResult object): The `ScenarioResult` for the `combined_stress` with no management actions applied.
# - `after_action` (ScenarioResult object): The `ScenarioResult` for the `combined_stress` with the `selected_combined_actions` applied.
# - `action_comparison` (DataFrame): A DataFrame comparing the impacts of different actions or action combinations.
# - `management_action_attribution` (DataFrame): A DataFrame showing the attributed impact of each management action on various metrics.
# - `management_action_traces` (list): Detailed traces of how management actions influenced the system.
# - Prints a confirmation message that calculations are complete.

Management strategy calculations completed. Detailed executive presentation follows below.


## MANAGEMENT RESPONSE DECISION LAB
**Purpose:** This section serves as a decision laboratory, analyzing the effectiveness of individual and combined management actions in mitigating the impact of the `combined_stress` scenario (from SECTION 8).
It compares the 'no action' baseline with specific management choices and a combined response.

**Analysis scope:** This section focuses on the combined-stress decision analysis, building upon the `management_strategies` object calculated in the previous cell (`73624830`).
- `NO ACTION`: Represents the control group (the stressed scenario without any management intervention).
- `BALANCED SINGLE-ACTION CHOICE`: Identifies the best single action based on an explicit equal-weight objective (though objective-specific winners are also highlighted).
- `COMBINED RESPONSE`: Applies several pre-selected actions simultaneously, re-running the Digital Twin with these actions for an integrated impact assessment.

The Prototype Risk Severity Score is a transparent management aid, not a regulatory risk score, which categorizes risks as Within Limit (= 0), Warning (= 1), or Critical (= 2) across configured metrics.



In [ ]:
# Purpose: This cell presents a comprehensive analysis of the bank's response to the 'combined_stress' scenario,
# detailing the impact of management actions, both individually and in combination.
# It visualizes the crisis without action, identifies best single actions, compares individual actions,
# shows multi-dimensional action value, presents the combined management response, analyzes risk severity transitions,
# highlights residual risks, and provides diagnostic notes.

# Input:
# - `management_strategies` (ManagementStrategyAnalysis object): The comprehensive object generated in the preceding cell (`73624830`),
#   containing all the results of the management strategy analysis, including `no_action`, `best_by_objective`,
#   `individual_comparison`, `multidimensional_action_value`, `strategy_comparison`, `severity_distribution`,
#   `severity_scores`, `threshold_status`, `residual_risk`, `unaddressed_risk_drivers`, `response_risk_flow`,
#   `action_efficiency`, `prioritisation_diagnostic`, `securities_sale_hqla_bridge`, and `interaction_rules`.

# Interdependencies:
# - Relies entirely on the `management_strategies` object, which is the output of cell `73624830`.
# - Uses visualization functions (`management_strategy_figure`, `severity_distribution_figure`,
#   `management_response_flow_figure`) imported from `digital_twin.visualizations` in SECTION 3.

print('A. CRISIS WITHOUT MANAGEMENT ACTION')
# Output: Displays the key metrics of the 'combined_stress' scenario before any management action is applied.
display(pd.DataFrame([management_strategies.no_action.metrics]))

print('B. BEST ACTIONS BY OBJECTIVE — no universal best action is claimed')
# Output: Displays a table showing the best individual management action for each specific objective (e.g., lowest loss, highest LCR).
display(management_strategies.best_by_objective)
# Output: Prints the name of the single management action identified as the 'balanced' choice.
print('BALANCED SINGLE-ACTION CHOICE →', management_strategies.selected_best_action)

print('C. INDIVIDUAL ACTION COMPARISON')
# Output: Displays a table comparing the impact of each individual management action on various metrics.
display(management_strategies.individual_comparison)

print('D. MULTI-DIMENSIONAL ACTION VALUE')
# Output: Displays a table assessing the value of actions across multiple dimensions, including non-monetized benefits.
display(management_strategies.multidimensional_action_value)
print('Operational resilience, customer, liquidity and other non-P&L benefits are shown separately and are not monetised unless explicitly modelled.')

print('E. COMBINED MANAGEMENT RESPONSE — one simultaneous Digital Twin rerun')
# Output: Prints the list of management actions combined into the selected response.
print(' + '.join(management_strategies.selected_combined_actions))
# Output: Displays a table comparing the 'no action' baseline with the combined management response on key metrics.
display(management_strategies.strategy_comparison)
# Output: Displays a visualization of the management strategy comparison.
management_strategy_figure(management_strategies.strategy_comparison).show()

print('F. RISK SEVERITY TRANSITION AND CONFIGURED THRESHOLDS')
# Output: Displays a table showing the distribution of risk severities (critical, warning, within limit) for various metrics.
display(management_strategies.severity_distribution)
# Output: Displays a table showing the raw severity scores.
display(management_strategies.severity_scores)
# Output: Displays a visualization of the risk severity distribution.
severity_distribution_figure(management_strategies.severity_distribution).show()
# Output: Displays a table showing the status of each risk threshold (e.g., breached, within).
display(management_strategies.threshold_status)
print('Warning and critical categories are mutually exclusive. A critical risk downgraded to warning is an improvement, even if the warning count rises.')

print('G. RESIDUAL RISK AFTER MANAGEMENT RESPONSE')
# Output: Displays a table summarizing the risks that remain after the combined management response.
display(management_strategies.residual_risk)
# Output: Displays a table listing any unaddressed risk drivers.
display(management_strategies.unaddressed_risk_drivers)
# Output: Displays a visualization of the management response flow, illustrating how actions mitigate risks.
management_response_flow_figure(management_strategies.response_risk_flow, management_strategies.selected_combined_actions).show()
# Output: Prints a concluding message about whether critical breaches were eliminated and if warning-level risks remain.
if not (management_strategies.residual_risk['Combined Severity'] == 'Critical').any():
    print('Combined management response eliminates configured critical breaches, but residual warning-level risks remain.')

print('H. ACTION-COST LIMITATIONS AND VALIDATION NOTES')
# Output: Displays a table showing the efficiency of management actions.
display(management_strategies.action_efficiency)
print('Action cost not modelled for most actions — economic ROI cannot be calculated.')
# Output: Displays diagnostic information regarding action prioritization.
display(management_strategies.prioritisation_diagnostic)
print('Prioritise Critical Payments raises processing capacity by 25%. Availability and customers use the underlying regional-capacity fraction, while nonlinear queue clearance reduces recovery from 12 to 7 hours. Classification B: correct explicit prioritisation and queue dynamics; no bug found.')
# Output: Displays the HQLA bridge related to securities sale, if applicable.
display(management_strategies.securities_sale_hqla_bridge)
print('Selling $2bn securities removes $1.7bn eligible securities after the 15% haircut and produces $1.96bn cash. After absorbing the negative cash gap, eligible cash rises $1.717bn, so total HQLA rises slightly by about $0.017bn. No double counting or HQLA bug found.')
# Output: Displays information about interaction rules between management actions.
display(management_strategies.interaction_rules)

A. CRISIS WITHOUT MANAGEMENT ACTION


,total_estimated_loss_bn,market_loss_bn,credit_loss_bn,operational_loss_bn,liquidity_pnl_loss_bn,cash_position_bn,hqla_bn,lcr,cet1_capital_bn,risk_weighted_assets_bn,...,customers_affected,applications_affected,recovery_time_hours,deposit_outflow_bn,cash_consumed_bn,hqla_consumed_bn,emergency_funding_requirement_bn,funding_cost_bn,asset_liquidation_bn,realised_asset_sale_loss_bn
0,0.816617,0.48,0.281286,0.054334,0.000997,-0.242533,15.3,1.029434,7.183383,51.84,...,1104499,5,12.0,10.223333,10.242533,10.0,0.242533,0.000997,0.0,0.0


B. BEST ACTIONS BY OBJECTIVE — no universal best action is claimed


,Objective,Best Action,Metric,Value,Direction
0,Loss reduction,Increase FX Hedge,total_estimated_loss_bn,0.696598,min
1,Liquidity (LCR),Draw Liquidity Facility,lcr,1.214966,max
2,Immediate cash,Draw Liquidity Facility,cash_position_bn,2.757467,max
3,Operational resilience,Activate Backup Region,Operational Severity Score,0.000000,min
4,Customer impact,Activate Backup Region,customers_affected,485980.000000,min
5,Balanced equal-weight resilience,Activate Backup Region,Balanced Severity Score,0.382654,min


BALANCED SINGLE-ACTION CHOICE → Activate Backup Region
C. INDIVIDUAL ACTION COMPARISON


...

,Metric,Before,After,Change
0,payment_availability,8.958333e-01,8.958333e-01,0.0
1,payment_backlog_bn,4.887560e-01,4.538203e-01,-0.034936
2,customers_affected,1.104499e+06,1.104499e+06,0
3,recovery_time_hours,1.200000e+01,7.000000e+00,-5.0
4,Validation classification,NaN,NaN,B. Correct consequence of explicit prioritisat...


Prioritise Critical Payments raises processing capacity by 25%. Availability and customers use the underlying regional-capacity fraction, while nonlinear queue clearance reduces recovery from 12 to 7 hours. Classification B: correct explicit prioritisation and queue dynamics; no bug found.


,Component,Before,Change,After
0,Raw Cash Position,-0.242533,1.960000,1.717467
1,Gross Securities Remaining,18.000000,-2.000000,16.000000
2,Securities Sold,0.000000,2.000000,2.000000
3,Realised Sale Proceeds,0.000000,1.960000,1.960000
4,Realised Sale Loss,0.000000,0.040000,0.040000
5,Eligible Cash,0.000000,1.717467,1.717467
6,Eligible Securities,15.300000,-1.700000,13.600000
7,Securities Haircut,2.700000,-0.300000,2.400000
8,Other HQLA,0.000000,0.000000,0.000000
9,Total HQLA,15.300000,0.017467,15.317467


Selling $2bn securities removes $1.7bn eligible securities after the 15% haircut and produces $1.96bn cash. After absorbing the negative cash gap, eligible cash rises $1.717bn, so total HQLA rises slightly by about $0.017bn. No double counting or HQLA bug found.


,Interaction,Rule
0,Backup activation + depositor contact,Backup changes simulated availability first; o...
1,Securities sale,Gross securities fall by the sale amount; cash...
2,Liquidity facility,Facility draw increases cash and its 30-day fu...
3,FX hedge,Hedge ratios alter market exposure only; opera...
4,Combined response,All actions are passed together to one engine ...


## SECTION 11 — AI Executive Explanation
**Purpose:** This section is designed to generate an executive-level explanation of the 'combined_stress' scenario's results, leveraging an AI (Large Language Model - LLM) if an API key is available. The explanation synthesizes various outputs from previous sections into a coherent narrative, focusing on key impacts, risks, and the effectiveness of management actions.

**Input:**
- **Pre-calculated JSON context:** The LLM receives a compact, validated JSON object that encapsulates critical information from the `combined_stress` scenario, its Monte Carlo diagnostics, management strategies, and residual risks. This includes baseline metrics, stressed metrics, LCR bridge details, Monte Carlo summary statistics, breach probabilities, management action attributions, and material assumptions.
- **OpenAI API Key (optional):** If `OPENAI_API_KEY` is set as an environment variable or Colab secret, an external LLM will be used to generate a more expanded and nuanced narrative. Otherwise, a deterministic Python summary is used.

**Output:**
- **Executive Summary:** A concise yet comprehensive narrative explaining the scenario's impact, key findings, and implications for the bank's resilience, tailored for executive audiences.
- **`executive_summary` (str):** The textual summary generated by the AI (or the deterministic fallback).
- **`section_15_payload` (dict):** A structured dictionary containing all the data passed to the AI for its explanation, also used for auditing and further analysis in Section 15.

**Interdependencies:**
- **`combined` (ScenarioResult object):** The output of SECTION 8 (Flagship Combined Stress), providing the core scenario results.
- **`mc_percentiles` and `mc_breach_probabilities` (DataFrames):** Outputs from SECTION 9 (Monte Carlo Simulation), providing probabilistic insights.
- **`management_action_attribution` and `management_strategies` (Objects/DataFrames):** Outputs from SECTION 10 (Management Actions), detailing the impact and analysis of management responses.
- **`lcr_bridge` (DataFrame):** Output from the LCR Validation section, providing a detailed breakdown of liquidity changes.
- **`mc_operational_diagnostics` (DataFrame):** Output from 'MONTE CARLO OPERATIONAL BREACH VALIDATION', explaining operational breach specifics.
- **`bank.assumptions` (dict):** Part of the initial `bank` object from SECTION 4, providing foundational assumptions.
- **LLM (`openai` library):** If the API key is available, this section interacts with the OpenAI API for text generation.

**Analysis scope: EXECUTIVE EXPLANATION OF `combined_stress`.** The context includes its LCR bridge, combined-stress Monte Carlo diagnostics, management strategies, and residual risks. It is not the eight-hour cloud executive summary.

Only compact, validated Python-calculated JSON is provided to the optional LLM. Without an API key, a deterministic Python summary is used.

In [ ]:
material_assumptions = {
    'prototype_only': True,
    'units': 'USD billions unless stated otherwise',
    'lcr_method': 'Simplified HQLA / stressed 30-day net cash outflows',
    'risk_weight_density': bank.assumptions['risk_weight_density'],
    'hqla_securities_haircut': bank.assumptions['hqla_securities_haircut'],
    'outage_withdrawal_response_rate': bank.assumptions['outage_withdrawal_response_rate'],
    'monte_carlo_runs': int(len(mc_results)),
    'monte_carlo_seed': 42,
}
executive_context = build_executive_context(
    combined, mc_percentiles, mc_breach_probabilities,
    management_action_attribution, material_assumptions,
    lcr_bridge=lcr_bridge,
    operational_monte_carlo_diagnostics=mc_operational_diagnostics,
    management_strategy_analysis=management_strategies,
)
executive_summary = explain_executive_context(executive_context)
print(executive_summary)

This combined stress scenario generated losses and operational disruptions impacting multiple risk domains, leading to several breaches in critical metrics and material customer impact. The detailed assessment and management response analysis follow.

---

**WHAT HAPPENED?**  
The flagship combined stress triggered material losses and liquidity strain—from market, credit, operational, and liquidity risk drivers. Total estimated loss reached approximately USD 0.82 billion, driven principally by market losses (0.48 billion) and credit losses (0.28 billion). Notably, cash position deteriorated into a critical shortfall (-0.24 billion), and payment availability dropped critically to 89.6%, with a 12-hour recovery time surpassing critical thresholds. The liquidity coverage ratio (LCR) fell into a warning status at 1.03, below the healthy boundary of 1.1, reflecting stressed liquidity needs.

---

**WHAT WAS AFFECTED?**  
- **Liquidity Position:** Cash and high-quality liquid assets (HQLA) d

## SECTION 12 — CEO DASHBOARD
**Analysis scope: FLAGSHIP `combined_stress` CEO/CIO VIEW.** The dashboard uses `combined`, its Monte Carlo breach probability, and `after_action` from the combined management response. It does not summarize every Section 7 scenario and does not show the independent cloud-eight-hour result unless the code is deliberately redirected.

#### Purpose
This cell constructs and displays the CEO Dashboard, a high-level summary designed for executive review. It aggregates critical metrics, Monte Carlo breach probabilities, top risk drivers, and the impact of management actions into a single, digestible view.

#### Input
*   `combined` (ScenarioResult object): From SECTION 8, providing the deterministic combined stress metrics.
*   `mc_results` (DataFrame): From SECTION 9, used to calculate the overall `breach_probability`.
*   `management_strategies` (ManagementStrategyAnalysis object): From SECTION 10, specifically `selected_best_action`, `selected_combined_actions`, `severity_scores`, and `threshold_status`.
*   `after_action` (ScenarioResult object): From SECTION 10, representing the bank's state after applying combined management actions.

#### Output
*   `ceo_dashboard` (DataFrame): A single-row DataFrame summarizing key metrics (loss, LCR, CET1, payment availability, backlog, customers affected, recovery time), breach probability, top risk drivers, selected best single action, combined response actions, and residual warning risks.
*   Displays the `ceo_dashboard` DataFrame.
*   Displays `management_strategies.severity_scores` and `management_strategies.threshold_status` DataFrames.
*   Displays a `bank_health_dashboard` visualization of `after_action.metrics`, providing a visual summary of the bank's health post-management actions.

#### Interdependencies
*   Relies on the `combined` object from SECTION 8.
*   Relies on `mc_results` from SECTION 9 to calculate `breach_probability`.
*   Relies on `management_strategies` and `after_action` objects from SECTION 10.
*   Uses `bank_health_dashboard` visualization function imported in SECTION 3.

In [ ]:
breach_probability = float(mc_results['risk_limit_breach'].mean())
drivers = sorted(['market_loss_bn','credit_loss_bn','operational_loss_bn'], key=lambda k: combined.metrics[k], reverse=True)
ceo_dashboard = pd.DataFrame([{'Scenario': combined.scenario, 'Total Loss': combined.metrics['total_estimated_loss_bn'], 'LCR': combined.metrics['lcr'], 'CET1 Ratio': combined.metrics['cet1_ratio'], 'Payment Availability': combined.metrics['payment_availability'], 'Payment Backlog': combined.metrics['payment_backlog_bn'], 'Customers Affected': combined.metrics['customers_affected'], 'Recovery Time': combined.metrics['recovery_time_hours'], 'Probability of Risk-Limit Breach': breach_probability, 'Top Risk Drivers': ', '.join(drivers), 'Balanced Single-Action Choice': management_strategies.selected_best_action, 'Combined Response': ' + '.join(combined_actions), 'Residual Warning Risks': ', '.join(management_strategies.residual_risk.loc[management_strategies.residual_risk['Combined Severity'] == 'Warning', 'Risk Metric'])}])
display(ceo_dashboard)
display(management_strategies.severity_scores)
display(management_strategies.threshold_status)
bank_health_dashboard(after_action.metrics).show()

,Scenario,Total Loss,LCR,CET1 Ratio,Payment Availability,Payment Backlog,Customers Affected,Recovery Time,Probability of Risk-Limit Breach,Top Risk Drivers,Balanced Single-Action Choice,Combined Response,Residual Warning Risks
0,Flagship combined stress,0.816617,1.029434,0.138568,0.895833,0.488756,1104499,12.0,1.0,"market_loss_bn, credit_loss_bn, operational_lo...",Activate Backup Region,Activate Backup Region + Prioritise Critical P...,"cash_position_bn, payment_availability"


,Strategy,Prototype Risk Severity Score,Severity Score Improvement vs No Action
0,No Action,7,0
1,Balanced Single Action,5,2
2,Combined Response,2,5


,Metric,Current Value,Warning Threshold,Critical Threshold,Direction,Severity,Distance to Warning Threshold / Healthy Boundary
0,lcr,1.381885,1.10,1.000,min,Within Limit,0.281885
1,cet1_ratio,0.141146,0.12,0.105,min,Within Limit,0.021146
2,cash_position_bn,3.799013,5.00,2.000,min,Warning,-1.200987
3,payment_availability,0.954167,0.98,0.950,min,Warning,-0.025833
4,payment_backlog_bn,0.225009,2.00,5.000,max,Within Limit,1.774991
5,total_estimated_loss_bn,0.683014,1.00,2.000,max,Within Limit,0.316986
6,recovery_time_hours,2.600000,4.00,8.000,max,Within Limit,1.400000


## SECTION 13 — Interactive CEO questions
**Default scope: `combined_stress`.** `ask_ceo(...)` explains the Section 8 `combined` result. Supplying `overrides` reruns `combined_stress` in Python before explanation. The question itself does not alter Sections 11, 12, 14, or 15.

#### Purpose
This cell defines the `ask_ceo` Python function, which serves as an interactive interface for querying the digital twin. It allows users to ask natural language questions about the `combined_stress` scenario's outcomes and can optionally re-run the scenario with specific parameter overrides.

#### Inputs
*   `question` (str): A natural language string representing the query for the AI explainer.
*   `overrides` (dict, optional): A dictionary containing parameters to temporarily override in the `combined_stress` scenario. If provided, the scenario is re-executed with these changes.

#### Outputs
*   `executive_summary` (str): A natural language explanation of the scenario results, generated by the `explain_results` function, tailored to the posed question.

#### Interdependencies
*   `engine` (ScenarioEngine object): Used internally to re-run the `combined_stress` scenario if `overrides` are provided. Initialized in SECTION 5.
*   `explain_results` (function): Imported from `digital_twin.ai_explainer` (in SECTION 3), this function generates the textual explanation.
*   `combined` (ScenarioResult object): If no `overrides` are specified, the function uses the `combined` object from SECTION 8 as its basis for explanation.

In [ ]:
def ask_ceo(question, overrides=None):
    if overrides:
        rerun = engine.run('combined_stress', overrides=overrides)
        return explain_results(rerun, question)
    return explain_results(combined, question)

print(ask_ceo('Why did liquidity deteriorate?'))
# Example rerun: ask_ceo('What happens if recovery takes 8 hours?', {'operational': {'recovery_time_hours': 8}})

Liquidity deteriorated primarily due to large deposit outflows totaling approximately 10.22 billion, which significantly reduced both the cash position and high-quality liquid assets (HQLA). Specifically, the cash position fell from an initial 10 billion to negative 0.24 billion, and HQLA decreased from 25.3 billion to about 15.3 billion. This led to a substantial decline in the Liquidity Coverage Ratio (LCR), dropping from a healthy 5.48 to a stressed 1.03, breaching the minimum warning threshold of 1.1.

The top propagation paths indicate that deposit outflows, driven by both corporate and retail segments, cascaded through liquidity position metrics to materially impact the LCR. The deposit outflows consumed nearly all available liquid resources and created an emergency funding requirement of 0.24 billion. Despite this, no asset liquidation or realized asset sale losses were recorded.

In summary, the deterioration in liquidity was caused by significant deposit withdrawals that exhau

## SECTION 14 — Export results
**Scope:** writes generated artifacts to `data/outputs`. Exports include the Section 7 scenario comparison plus flagship combined-stress, Monte Carlo, propagation, and management-action outputs. This section does not run a new scenario.

In [ ]:
output_dir = pathlib.Path(PROJECT_ROOT) / 'data' / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
scenario_table.to_csv(output_dir / 'scenario_results.csv', index=False)
mc_results.to_csv(output_dir / 'monte_carlo_results.csv', index=False)
action_comparison.to_csv(output_dir / 'management_action_comparison.csv', index=False)
with (output_dir / 'executive_summary.json').open('w', encoding='utf-8') as handle:
    json.dump({'scenario': after_action.to_dict(), 'summary': executive_summary, 'monte_carlo_breach_probability': breach_probability}, handle, indent=2)
print('Exports written to', output_dir)

Exports written to /content/drive/MyDrive/ai-financial-digital-twin/data/outputs


## SECTION 15 — OpenAI input, scenario, and output summary
**Scope:** final audit and narrative package. It catalogues configured scenario inputs and deterministic outputs, but the detailed Monte Carlo, management-response, CEO conclusions, and `section_15_payload` are centred on the flagship `combined_stress` workflow.

The OpenAI LLM only summarizes already-calculated values; it does not perform financial calculations. Set `OPENAI_API_KEY` as a Colab secret or environment variable to enable the optional narrative.

In [ ]:
import os
if not os.getenv('OPENAI_API_KEY'):
    try:
        from google.colab import userdata
        colab_key = userdata.get('OPENAI_API_KEY')
        if colab_key:
            os.environ['OPENAI_API_KEY'] = colab_key
    except (ImportError, KeyError):
        pass
print('OPENAI_API_KEY available:', bool(os.getenv('OPENAI_API_KEY')))

OPENAI_API_KEY available: True


In [ ]:
from digital_twin.config import load_baseline, load_risk_limits, load_scenario

# Convert DataFrames through JSON so NumPy scalar types become API-safe Python values.
def json_records(frame):
    return json.loads(frame.to_json(orient='records'))

all_scenario_names = scenario_names + ['combined_stress']
all_scenario_results = individual_results + [combined]
scenario_catalogue = []
for scenario_id, scenario_result in zip(all_scenario_names, all_scenario_results):
    scenario_catalogue.append({
        'scenario_id': scenario_id,
        'definition': load_scenario(scenario_id),
        'calculated_metrics': scenario_result.metrics,
        'risk_limit_breaches': scenario_result.risk_limit_breaches,
        'propagation_paths': scenario_result.propagation_paths,
    })

section_15_payload = {
    'prototype_disclaimer': 'Synthetic prototype only; not a regulatory model or financial advice.',
    'input_summary': {
        'baseline_configuration': load_baseline(),
        'risk_limits': load_risk_limits(),
        'balance_sheet': json_records(bank.balance_sheet),
        'customer_segments': json_records(bank.customer_segments),
        'fx_exposures': json_records(bank.fx_exposures),
        'counterparty_summary': {
            'count': int(len(bank.counterparties)),
            'total_ead_bn': float(bank.counterparties['ead_bn'].sum()),
            'ratings': bank.counterparties['rating'].value_counts().to_dict(),
            'sectors': bank.counterparties['sector'].value_counts().to_dict(),
        },
        'operating_model_counts': {
            'applications': int(len(bank.applications)),
            'vendors': int(len(bank.vendors)),
            'infrastructure_nodes': int(len(bank.infrastructure)),
            'payment_systems': int(len(bank.payment_systems)),
            'dependency_edges': int(len(bank.dependencies)),
        },
        'random_seed': int(bank.seed),
    },
    'scenario_summary': scenario_catalogue,
    'output_summary': {
        'baseline_metrics': baseline,
        'flagship_combined_stress': combined.to_dict(),
        'monte_carlo': {
            'runs': int(len(mc_results)),
            'statistics': json_records(mc_summary),
            'probability_any_risk_limit_breach': float(breach_probability),
        },
        'management_actions_applied': combined_actions,
        'management_action_comparison': json_records(action_comparison),
        'post_action_metrics': after_action.metrics,
        'post_action_breaches': after_action.risk_limit_breaches,
    },
}

# Keep the detailed object for local comparison, but send only compact validated context.
raw_section_15_json = json.dumps(section_15_payload, allow_nan=False)
section_15_payload = executive_context
section_15_json = json.dumps(section_15_payload, indent=2, allow_nan=False)
print(f'Raw simulation payload size: {len(raw_section_15_json.encode("utf-8")):,} bytes')
print(f'Executive context payload size: {len(section_15_json.encode("utf-8")):,} bytes')
if len(section_15_json.encode('utf-8')) > 60000:
    raise ValueError('Executive context exceeds the 60,000-byte API safety limit.')

if os.getenv("OPENAI_API_KEY"):
    from openai import OpenAI

    client = OpenAI()

    response = client.responses.create(
        model=os.getenv("OPENAI_MODEL", "gpt-4.1-mini"),

        instructions="""
You are an executive banking risk narrator explaining results from a
synthetic AI Financial Digital Twin prototype.

The supplied JSON is the ONLY source of truth.

Your role is to INTERPRET validated simulation results for senior banking
management. You are not a calculation engine.

STRICT GROUNDING RULES

1. Use only facts explicitly contained in the supplied validated JSON.

2. Never invent, estimate, interpolate, derive, recalculate, or correct
   numerical values.

3. Never independently calculate ratios, percentages, breach counts,
   severity scores, losses, improvements, probabilities, or thresholds.

4. Never override deterministic classifications produced by the Python
   simulation.

5. Clearly distinguish:
   - configured inputs;
   - scenario assumptions;
   - deterministic calculated outputs;
   - Monte Carlo simulation results;
   - management-action effects;
   - residual risks.

6. Never describe a WARNING metric as healthy or within limit.

7. Never state that "all breaches are eliminated" when warning breaches
   remain.

8. Distinguish explicitly between:
   - Warning breaches
   - Critical breaches
   - Prototype Risk Severity Score

9. The Prototype Risk Severity Score is a non-regulatory management
   indicator. Never describe it as a regulatory risk score.

10. For Monte Carlo results use wording such as:
       "X% of simulated runs..."
    rather than:
       "X% chance..."
    This prototype does not estimate real-world probabilities.

11. Never say a metric "doubled", "halved", "increased by X%" or
    "decreased by X%" unless that exact comparison is explicitly supplied
    in the validated JSON.

12. Never infer causality solely from correlation between output metrics.
    Only describe causal relationships explicitly represented in the
    supplied simulation context.

13. Never describe CET1 improvement as "improved capital quality".
    Refer instead to stressed CET1 capital or the stressed CET1 ratio.

14. For management actions:
    - use the exact action names supplied;
    - do not claim a universal best action;
    - distinguish best actions by management objective;
    - describe the balanced single-action choice separately;
    - list the exact actions included in the combined response.

15. Never call the combined response "all actions" unless the supplied
    context explicitly states that every available action is included.

16. Management actions must not be described as additive if the supplied
    context states that they were simulated simultaneously.

17. Clearly identify residual WARNING and CRITICAL risks after management
    actions.

18. Clearly identify material risk drivers that remain unchanged or
    unaddressed.

19. Do not claim that a management action is economically optimal.

20. Do not calculate or claim economic ROI where action costs or benefits
    are not fully modelled.

21. Never claim:
    - regulatory compliance;
    - regulatory capital adequacy;
    - regulatory LCR compliance;
    - production readiness;
    - predictive accuracy;
    - real-world probability;
    - economic optimality.

22. Always state that this is a synthetic, simplified, non-regulatory
    prototype intended to demonstrate Digital Twin feasibility.

23. If information required to support a conclusion is absent from the
    JSON, say that it is not available from the prototype rather than
    inferring it.

EXECUTIVE INTERPRETATION PRINCIPLE

Focus on:
    Shock
      → propagation
      → financial / operational consequence
      → management intervention
      → resulting improvement
      → residual risk

Explain why the result matters to bank management, but never create facts
that are not present in the validated simulation output.
""",

        input=(
            """
Using only VALIDATED_EXECUTIVE_CONTEXT below, produce a concise
executive banking risk report.

Use the following headings exactly:

1. Prototype Scope and Disclaimer

2. Scenario Being Tested

3. Baseline Position

4. Stress Impact

5. How Risk Propagated Through the Bank

6. Monte Carlo Range and Breach Frequency

7. Best Management Actions by Objective

8. Combined Management Response

9. Residual Risks

10. Executive Conclusions


REPORTING REQUIREMENTS

Prototype Scope and Disclaimer
- Clearly state synthetic, simplified and non-regulatory nature.
- State that numerical calculations come from deterministic Python
  simulation components, not from the LLM.

Scenario Being Tested
- Describe the actual stress scenario and major shocks.
- Do not describe scenarios that were not executed as if they occurred.

Baseline Position
- Include only the most decision-relevant baseline metrics.

Stress Impact
Prioritise:
- total estimated loss;
- major loss components;
- cash position;
- HQLA/LCR;
- CET1 ratio;
- payment availability;
- payment backlog;
- customers affected;
- recovery time.

Clearly identify warning and critical breaches using supplied
classifications.

How Risk Propagated Through the Bank
Explain only propagation paths explicitly supplied by the Digital Twin.

Where supported, describe patterns such as:

Infrastructure
→ Application
→ Business Service
→ Customer
→ Deposit Behaviour
→ Liquidity
→ Risk Metric

Do not invent dependencies.

Monte Carlo Range and Breach Frequency
- Report important P5/median/P95 ranges where supplied.
- Say "percentage of simulated runs", not real-world probability.
- Distinguish warning and critical thresholds.

Best Management Actions by Objective
Do NOT claim one universal best action.

Where supplied, identify separately:
- Loss reduction
- Liquidity/LCR
- Immediate cash
- Operational resilience
- Customer impact
- Balanced resilience

Explain the trade-offs between actions.

Combined Management Response
- List the exact actions included.
- State that they were simulated together if supplied by the context.
- Compare the combined response with no action.
- Do not add individual action benefits together.

Residual Risks
Explicitly identify:
- remaining warning breaches;
- remaining critical breaches;
- Prototype Risk Severity Score where supplied;
- materially unchanged/unaddressed risk drivers.

Executive Conclusions
Answer four management questions:

1. What is the most important vulnerability exposed by the stress?
2. Which individual actions are strongest for which objectives?
3. What does the combined response achieve?
4. What material risks remain?

Do not introduce recommendations that were not tested by the Digital Twin.
You may state that additional scenarios or management actions would need
to be simulated.

Avoid unnecessary repetition.

VALIDATED_EXECUTIVE_CONTEXT:

"""
            + section_15_json
        )
    )
    section_15_summary = response.output_text
else:
    section_15_summary = (
        'OPENAI_API_KEY was not found in the Colab environment. The structured Section 15 payload was created successfully, '
        'but the LLM narrative was skipped. Set the environment variable and rerun this cell.'
    )

print(section_15_summary)
with (output_dir / 'openai_full_run_summary.json').open('w', encoding='utf-8') as handle:
    json.dump({'validated_input': section_15_payload, 'llm_summary': section_15_summary}, handle, indent=2, allow_nan=False)
with (output_dir / 'openai_full_run_summary.md').open('w', encoding='utf-8') as handle:
    handle.write(section_15_summary)
print('Section 15 exports written to', output_dir)


Raw simulation payload size: 69,460 bytes
Executive context payload size: 49,417 bytes
1. Prototype Scope and Disclaimer

This report is based on a synthetic, simplified, and non-regulatory prototype designed to demonstrate the feasibility of a Financial Digital Twin for banking risk assessment. All numerical calculations and outputs are generated deterministically by Python simulation components within the prototype, and no independent calculations, interpolations, or external inferences have been made beyond what is explicitly supplied.


2. Scenario Being Tested

The executed scenario is the "Flagship combined stress," which incorporates multiple shocks including market shocks (notably USD and volatility shocks), credit shocks from a major counterparty default, and operational disruptions affecting cloud infrastructure and payment systems. These shocks generate losses, deposit outflows, and operational interruptions.


3. Baseline Position

Prior to stress, the bank shows no losses,